In [ ]:
# =====================================================================
# PARAMÈTRES  (tous les réglages sont regroupés dans ce bloc)
# =====================================================================
# ---- Simulation ----
CAPACITE_MODE = "Finie"       # "Finie" ou "Infinie"
MODE_PLANIF   = "Plutard"       # "Plutot" ou "Plutard"
SIM_DESC      = "SIM_Forcast_10_2026_FIN"
SIM_USER      = "ADMIN"
SIM_LAUNCH_DATE = "2026-10-26"  # date de lancement "YYYY-MM-DD" (None = date du jour)
                                # Valable pour TOUTES les simulations (Finie et Infinie) :
                                # aucun OF n'est planifie avant le lundi de cette semaine.

# ---------------------------------------------------------------------
# ARRETS SPECIFIQUES A CETTE SIMULATION (periodes ou RIEN n'est planifie, tous postes)
# Liste de (date_debut, date_fin) inclusives, format "YYYY-MM-DD". Liste vide [] = aucun arret.
# ---------------------------------------------------------------------
SIM_ARRETS = [
    # ("2027-08-02", "2027-08-23"),
]

# ---------------------------------------------------------------------
# VERIFICATIONS  (True = verification ACTIVE | False = ignoree | None = defaut du mode)
# Defaut (None) : toutes ACTIVES en capacite "Finie", toutes IGNOREES en "Infinie".
# Chaque verification se regle independamment, dans les deux modes.
# ---------------------------------------------------------------------
VERIF_STOCK       = True   # composants disponibles (lots produits par les OF enfants)
VERIF_ECART_OPS   = None   # ecart max entre 2 operations d'un meme OF (MAX_GAP_BETWEEN_OPS_HOURS)
VERIF_FENETRE     = None   # fenetre avance/retard par article (CST1_0/CST2_0)
                           # ! Reglable uniquement en "Infinie" : en "Finie" elle reste toujours active.
# ---------------------------------------------------------------------
# VALEURS PAR DEFAUT DE PLANIFICATION (durees en JOURS, si absentes de X3ITM)
# ---------------------------------------------------------------------
HORIZON_H_DAYS = 70               # horizon de groupage par defaut (jours)
ADVANCE_RETREAT_DAYS = 21         # avance/retard par defaut pour la planif (jours)
MAX_GAP_BETWEEN_OPS_HOURS = 336   # ecart max entre 2 operations d'un meme OF

# ---------------------------------------------------------------------
# EVOLUTION DE LA QUANTITE A FABRIQUER PAR ARTICLE (scenario de demande)
# Pourcentage applique a la quantite de TOUS les OF de l'article (horizon filtre),
# et aux heures de ses operations (ChargeCalculee) dans la meme proportion.
#   10 = +10 %   |   -5 = -5 %   |   dictionnaire vide {} = aucune evolution
# ---------------------------------------------------------------------
EVOL_QTE_PAR_ARTICLE = {
    # "PF001234": 10,
    # "PF005678": -5,
}
EVOL_QTE_TABLE       = None   # ex "SIM_EVOL_QTE" (colonnes Article, Pct) : lue en plus du dictionnaire
                              # (en cas de doublon, la valeur du dictionnaire ci-dessus l'emporte)
EVOL_QTE_PROPAGER_SF = True   # True = augmente aussi les OF des composants (SF) du surplus reellement
                              # consomme via la nomenclature (evite les faux "stock insuffisant")
EVOL_QTE_ARRONDI     = True   # True = quantite arrondie a l'unite superieure

# ---------------------------------------------------------------------
# FILTRAGE PAR PLAGE DE DATES DE BESOIN (format "YYYY-MM-DD" ou None)
# ---------------------------------------------------------------------
HORIZON_DEB = "2026-10-26"
HORIZON_FIN = "2028-03-20"

# ---------------------------------------------------------------------
# LOGS
# ---------------------------------------------------------------------
LOG_LEVEL = "detailed"   # 'detailed' genere les tables Delta ; 'none' desactive tout
LOG_FILTER = "all"       # "all" | "DIVC" | "none"  <- MODIFIEZ ICI selon vos besoins
LOG_WRITE_MODE = "overwrite"

# ---------------------------------------------------------------------
# LAKEHOUSE
# ---------------------------------------------------------------------
LAKEHOUSE_NAME = "Planning_Lakehouse"

# ---------------------------------------------------------------------
# IMPORTS ET VALEURS DERIVEES
# ---------------------------------------------------------------------
from datetime import datetime, timedelta, time, date
from collections import defaultdict, deque
from functools import lru_cache
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, DateType
)
import re
import os

# VALEURS DERIVEES des parametres (ne pas modifier ici, voir le bloc PARAMETRES en haut)
WEEK_SORT_ORDER = "earliest" if MODE_PLANIF == "Plutot" else "closest"

# ---------------------------------------------------------------------
# PLANCHER "PAS DE PASSE" : semaine (lundi) de lancement de la simulation.
# Quel que soit MODE_PLANIF ("Plutot"/avance ou "Plutard"/retard), aucun OF
# ne doit jamais etre planifie a une semaine anterieure a celle du lancement.
# Ex : date de lancement 17/09/2026 (jeudi) -> lundi de lancement = 14/09/2026.
# La date se regle en haut du notebook (SIM_LAUNCH_DATE) ; elle s'applique aussi
# bien en capacite Finie qu'Infinie.
# ---------------------------------------------------------------------
_launch_dt = datetime.strptime(SIM_LAUNCH_DATE, "%Y-%m-%d") if SIM_LAUNCH_DATE else datetime.now()
SIM_LAUNCH_MONDAY = datetime.combine(
    (_launch_dt - timedelta(days=_launch_dt.weekday())).date(), time.min
)


def push_out_of_arrets(dt):
    """Repousse dt a la fin de la periode d'arret (SIM_ARRETS) qui le contient, le cas echeant.
    Sert a la planification en capacite infinie 'simple', ou l'OF est place directement a sa
    date de besoin sans passer par le calendrier des postes."""
    periods = []
    for _d1, _d2 in SIM_ARRETS:
        try:
            periods.append((try_parse_date(_d1).replace(hour=0, minute=0, second=0),
                            try_parse_date(_d2).replace(hour=23, minute=59, second=59)))
        except ValueError:
            pass
    moved = True
    while moved:
        moved = False
        for un_start, un_end in periods:
            if un_start <= dt <= un_end:
                dt = (un_end + timedelta(minutes=1)).replace(second=0, microsecond=0)
                moved = True
    return dt

# Noms des tables de sortie pour les logs
OUT_FAILURE_DETAILS = "S_failure_details"
OUT_FAILURE_SUMMARY_TYPOLOGIE = "S_failure_summary_typologie"
OUT_FAILURE_SUMMARY_TOP = "S_failure_summary_top"

# Ces variables sont definies dynamiquement apres detection du prefixe
TBL_OF_WORK = "sim_of_work"
TBL_BOM = "sim_nom_child"
TBL_POSTE_CAPA = "SIM_POSTE_CAPA"
TBL_X3OF = "X3OF"
TBL_X3ITM = "X3ITM"
OUT_GROUPED_NEEDS = "S_grouped_needs"
OUT_SMOOTHING = "S_smoothing"
OUT_SMOOTHING_OPS = "S_smoothing_operations"
OUT_WEEKLY_CAPACITY = "S_weekly_capacity"
OUT_GROUPS_ANCHORS = "S_groups_anchors"


# ---------------------------------------------------------------------
# UTILITAIRES
# ---------------------------------------------------------------------
def norm_code(x) -> str:
    if x is None:
        return ""
    s = str(x).replace("\ufeff", "")
    s = "".join(s.split())
    return s.upper()


def _norm(x) -> str:
    if x is None:
        return ""
    return ''.join(c for c in str(x).upper() if c.isalnum())


def try_parse_date(value):
    if value is None:
        raise ValueError("Cannot parse None as date.")
    if isinstance(value, datetime):
        return value
    if isinstance(value, date):
        return datetime.combine(value, time.min)
    text = str(value).strip()
    if not text:
        raise ValueError("Cannot parse empty string as date.")
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d/%m/%y", "%m/%d/%Y", "%Y/%m/%d", "%d-%m-%Y", "%d.%m.%Y"):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue
    raise ValueError(f"Date format for '{text}' not recognized.")


def try_parse_float(value):
    if isinstance(value, (int, float)):
        return float(value)
    if value is None:
        raise ValueError("Cannot parse None as float.")
    text = str(value).strip()
    if not text:
        raise ValueError("Cannot parse empty string as float.")
    sanitized = text.replace("\u00a0", "").replace(" ", "")
    if sanitized.count(",") > 0 and sanitized.count(".") > 0:
        sanitized = sanitized.replace(".", "")
    sanitized = sanitized.replace(",", ".")
    return float(sanitized)


@lru_cache(maxsize=None)
def find_qty_of_component_in_product_cached(product_to_make_id, component_to_find_id, bom_tuple):
    if product_to_make_id == component_to_find_id:
        return 1.0
    total = 0.0
    for parent, child, qty in bom_tuple:
        if parent == product_to_make_id:
            total += qty * find_qty_of_component_in_product_cached(child, component_to_find_id, bom_tuple)
    return total


# ---------------------------------------------------------------------
# CLASSES METIER
# ---------------------------------------------------------------------
class ManufacturingOrder:
    __slots__ = (
        'id', 'designation', 'product_id', 'raw_product_id', 'product_type',
        'bom_level', 'need_date', 'original_need_date', 'source_qty', 'quantity', 'unit',
        'horizon_days', 'retard_days', 'advance_days',
        'assigned_group_id', 'status',
        'scheduled_start_date', 'scheduled_end_date', 'fg', 'cat', 'us', 'fs',
        'individual_product_stock', 'normalized_id', 'normalized_product',
        'priority', 'num_plan', 'division', 'category',
        'qty_origine', 'coef_evol'
    )

    def __init__(
        self, id, designation, product_id, product_type, bom_level, need_date_str,
        quantity, fg, cat, us, fs, horizon_days=None, retard_days=None,
        advance_days=None, unit="U", status="UNASSIGNED", priority=None, category=None
    ):
        self.id = id
        self.designation = designation
        self.raw_product_id = product_id
        self.product_id = _norm(product_id)
        self.product_type = product_type
        self.bom_level = int(bom_level)
        self.need_date = try_parse_date(need_date_str)
        self.original_need_date = self.need_date   # copie de la date brute
        self.source_qty = str(quantity) if quantity is not None else ""
        self.quantity = try_parse_float(quantity)
        self.unit = unit

        self.horizon_days = horizon_days
        self.retard_days = retard_days
        self.advance_days = advance_days

        self.assigned_group_id = None
        self.status = "UNASSIGNED"
        self.scheduled_start_date = None
        self.scheduled_end_date = None
        self.fg = fg
        self.cat = cat
        self.us = us
        self.fs = fs
        self.division = None
        self.individual_product_stock = 0.0
        self.normalized_id = _norm(self.id)
        self.normalized_product = self.product_id
        self.priority = priority
        self.num_plan = None
        self.category = category
        self.qty_origine = self.quantity   # quantite X3 avant evolution (EVOL_QTE_*)
        self.coef_evol = 1.0               # coefficient applique (1.10 = +10 %)

    def __repr__(self):
        return (
            f"OF(id={self.id}, prod='{self.raw_product_id}', type='{self.product_type}', "
            f"level={self.bom_level}, need={self.need_date.strftime('%Y-%m-%d')}, "
            f"orig={self.original_need_date.strftime('%Y-%m-%d')}, "
            f"qty={self.quantity}, div='{self.division}', group={self.assigned_group_id}, "
            f"status='{self.status}', cat='{self.category}')"
        )


class BOMEntry:
    __slots__ = ('parent_product_id', 'child_product_id', 'quantity_child_per_parent',
                 'child_bom_level', 'parent_bom_level', 'raw_parent', 'raw_child')

    def __init__(self, parent_product_id, child_product_id, quantity_child_per_parent,
                 child_bom_level, parent_bom_level=None):
        self.raw_parent = parent_product_id
        self.raw_child = child_product_id
        self.parent_product_id = _norm(parent_product_id)
        self.child_product_id = _norm(child_product_id)
        self.quantity_child_per_parent = try_parse_float(quantity_child_per_parent)
        try:
            self.child_bom_level = int(child_bom_level) if child_bom_level not in (None, "") else 0
        except Exception:
            self.child_bom_level = 0
        if parent_bom_level not in (None, ""):
            try:
                self.parent_bom_level = int(parent_bom_level)
            except Exception:
                self.parent_bom_level = 0
        else:
            self.parent_bom_level = 0

    def __repr__(self):
        return (f"BOM(parent='{self.raw_parent}' uses {self.quantity_child_per_parent} "
                f"of child='{self.raw_child}')")


class Group:
    __slots__ = (
        'id', 'ps_product_id', 'raw_ps_product_id', 'time_window_start', 'time_window_end',
        'ofs', 'current_ps_stock_available', 'component_stocks', 'individual_product_stocks',
        'product_consumption', 'ofs_by_id', 'ofs_by_product'
    )

    def __init__(self, id, ps_product_id, initial_ps_of, window_start_date, window_end_date,
                 initial_ps_as_stock=False):
        self.id = id
        self.raw_ps_product_id = ps_product_id
        self.ps_product_id = _norm(ps_product_id)
        self.time_window_start = window_start_date
        self.time_window_end = window_end_date
        self.ofs = []
        self.ofs_by_id = {}
        self.ofs_by_product = defaultdict(list)
        self.current_ps_stock_available = 0
        self.component_stocks = defaultdict(float)
        self.individual_product_stocks = defaultdict(float)
        self.product_consumption = defaultdict(float)
        ps_q = initial_ps_of.quantity if initial_ps_as_stock else 0
        self.add_of(initial_ps_of, ps_quantity_change=ps_q)

    def add_of(self, of_to_add, ps_quantity_change=0):
        self.ofs.append(of_to_add)
        self.ofs_by_id[of_to_add.id] = of_to_add
        self.ofs_by_product[of_to_add.product_id].append(of_to_add)
        self.current_ps_stock_available += ps_quantity_change
        self.component_stocks[of_to_add.product_id] += ps_quantity_change
        self.individual_product_stocks[of_to_add.product_id]
        self.product_consumption[of_to_add.product_id]
        of_to_add.individual_product_stock = 0.0
        of_to_add.assigned_group_id = self.id
        of_to_add.status = "ASSIGNED"

    def calculate_consumption(self, bom_data):
        if not self.ofs:
            self.component_stocks.clear()
            self.individual_product_stocks.clear()
            self.product_consumption.clear()
            return

        product_level = {}
        for bom in bom_data:
            p, c, cl, pl = bom.parent_product_id, bom.child_product_id, bom.child_bom_level, bom.parent_bom_level
            if c and cl > 0:
                product_level[c] = max(product_level.get(c, 0), cl)
            if p and pl > 0:
                product_level[p] = max(product_level.get(p, 0), pl)

        produced_qty = defaultdict(float)
        for of in self.ofs:
            produced_qty[of.product_id] += of.quantity
        if not produced_qty:
            self._clear_and_reset()
            return
        for pid in produced_qty:
            if pid not in product_level:
                product_level[pid] = 0

        levels = set(product_level[pid] for pid in produced_qty)
        max_level, min_level = max(levels), min(levels)

        bom_by_parent = defaultdict(list)
        for bom in bom_data:
            if bom.parent_product_id in produced_qty:
                bom_by_parent[bom.parent_product_id].append((bom.child_product_id, bom.quantity_child_per_parent))

        product_stock = defaultdict(float)
        product_consumption = defaultdict(float)
        for lvl in range(max_level, min_level - 1, -1):
            for pid, plvl in product_level.items():
                if plvl != lvl or pid not in produced_qty:
                    continue
                qty_prod = produced_qty[pid]
                if qty_prod <= 0:
                    continue
                product_stock[pid] += qty_prod
                for child_id, q_child in bom_by_parent.get(pid, []):
                    need = qty_prod * q_child
                    product_stock[child_id] -= need
                    product_consumption[child_id] += need

        remaining_per_of = {of.id: 0.0 for of in self.ofs}
        for prod_norm in produced_qty:
            stock_left = max(0.0, product_stock.get(prod_norm, 0.0))
            ofs_sorted = sorted(self.ofs_by_product.get(prod_norm, []), key=lambda o: (o.need_date, o.id))
            for of in ofs_sorted:
                if stock_left <= 0:
                    break
                assign = min(stock_left, of.quantity)
                remaining_per_of[of.id] = assign
                stock_left -= assign

        for of in self.ofs:
            of.individual_product_stock = max(0.0, remaining_per_of.get(of.id, 0.0))

        self.product_consumption = dict(product_consumption)
        self.component_stocks = dict(product_stock)
        self.individual_product_stocks = {
            pid: sum(of.individual_product_stock for of in self.ofs_by_product.get(pid, []))
            for pid in produced_qty
        }

    def _clear_and_reset(self):
        self.component_stocks.clear()
        self.individual_product_stocks.clear()
        self.product_consumption.clear()
        for of in self.ofs:
            of.individual_product_stock = 0.0


class Post:
    __slots__ = (
        'id', 'name', 'is_bottleneck', 'work_start_time', 'work_end_time',
        'lunch_start_time', 'lunch_end_time', 'daily_capacity_hours',
        'weekly_capacity_by_week', 'weekly_load_hours', 'weekly_remaining_capacity',
        'override_weeks', 'unavailable_periods', 'scheduled_slots', 'slots_by_week',
        'planning_start_monday', '_max_defined_week', '_first_week_index', '_last_week_index',
        '_first_week_date', '_last_week_date'
    )

    def __init__(self, id, name, weekly_capacity_by_week=None, is_bottleneck=True, override_weeks=None):
        self.id = id
        self.name = name
        self.is_bottleneck = bool(is_bottleneck)
        self.work_start_time = time(0, 0)
        self.work_end_time = time(23, 59)
        self.lunch_start_time = time(0, 0)
        self.lunch_end_time = time(0, 0)
        self.daily_capacity_hours = 24.0
        self.override_weeks = set()
        self.weekly_capacity_by_week = dict(weekly_capacity_by_week) if weekly_capacity_by_week else {}
        self.weekly_load_hours = defaultdict(float)
        self.weekly_remaining_capacity = defaultdict(float)
        for wk, cap in self.weekly_capacity_by_week.items():
            self.weekly_remaining_capacity[wk] = float(cap)
        self.unavailable_periods = []
        self.scheduled_slots = []
        self.slots_by_week = defaultdict(list)
        self.planning_start_monday = None
        self._max_defined_week = max(self.weekly_capacity_by_week.keys()) if self.weekly_capacity_by_week else 0
        self._first_week_index = None
        self._last_week_index = None
        self._first_week_date = None
        self._last_week_date = None

    def set_week0_monday(self, monday_date):
        self.planning_start_monday = monday_date
        if self.weekly_capacity_by_week:
            self._first_week_index = min(self.weekly_capacity_by_week.keys())
            self._last_week_index = max(self.weekly_capacity_by_week.keys())
            self._first_week_date = datetime.combine(
                self.planning_start_monday + timedelta(days=7 * (self._first_week_index - 1)), time.min)
            self._last_week_date = datetime.combine(
                self.planning_start_monday + timedelta(days=7 * self._last_week_index), time.min)

    def _ensure_planning_start(self, dt_obj):
        if self.planning_start_monday is None:
            d = dt_obj.date()
            self.planning_start_monday = d - timedelta(days=d.weekday())
            if self.weekly_capacity_by_week:
                self._first_week_index = min(self.weekly_capacity_by_week.keys())
                self._last_week_index = max(self.weekly_capacity_by_week.keys())
                self._first_week_date = datetime.combine(
                    self.planning_start_monday + timedelta(days=7 * (self._first_week_index - 1)), time.min)
                self._last_week_date = datetime.combine(
                    self.planning_start_monday + timedelta(days=7 * self._last_week_index), time.min)

    def _relative_week_index(self, dt_obj):
        if self.planning_start_monday is None:
            self._ensure_planning_start(dt_obj)
        delta_days = (dt_obj.date() - self.planning_start_monday).days
        return 1 + (delta_days // 7)

    def _week_key(self, dt_obj):
        return self._relative_week_index(dt_obj)

    def _allowed_hours_for_week(self, dt_obj):
        w = self._relative_week_index(dt_obj)
        return float(self.weekly_capacity_by_week.get(w, 0.0))

    def _weekly_hours_distribution(self, start_dt, end_dt):
        if end_dt is None:
            return defaultdict(float)
        dist = defaultdict(float)
        current = start_dt
        while current < end_dt:
            wk = self._week_key(current)
            monday_of_week = self.planning_start_monday + timedelta(days=7 * (wk - 1))
            week_end_dt = datetime.combine(monday_of_week + timedelta(days=7), time.min)
            segment_end = min(end_dt, week_end_dt)
            hours = (segment_end - current).total_seconds() / 3600.0
            dist[wk] += hours
            current = segment_end
        return dist

    def _has_any_capacity_in_week(self, dt_obj):
        wk = self._relative_week_index(dt_obj)
        constrained = self.is_bottleneck or (wk in self.override_weeks)
        if not constrained:
            return True
        if self._first_week_index is not None and wk < self._first_week_index:
            return False
        if self._last_week_index is not None and wk > self._last_week_index:
            return False
        remaining = self.weekly_remaining_capacity.get(wk, 0.0)
        return remaining > 1e-6

    def _can_add_slot_in_week(self, start_dt, end_dt):
        if end_dt is None:
            return False
        dist = self._weekly_hours_distribution(start_dt, end_dt)
        constrained_weeks = {wk for wk in dist.keys() if self.is_bottleneck or (wk in self.override_weeks)}
        if not constrained_weeks:
            return True
        for wk, add_hours in dist.items():
            if wk not in constrained_weeks:
                continue
            if self._first_week_index is not None and wk < self._first_week_index:
                return False
            if self._last_week_index is not None and wk > self._last_week_index:
                return False
            remaining = self.weekly_remaining_capacity.get(wk, 0.0)
            if add_hours > remaining + 1e-6:
                return False
        return True

    def _register_slot_in_week_load(self, start_dt, end_dt):
        dist = self._weekly_hours_distribution(start_dt, end_dt)
        for wk, hours in dist.items():
            self.weekly_load_hours[wk] += hours
            if self.is_bottleneck or (wk in self.override_weeks):
                self.weekly_remaining_capacity[wk] = max(0.0, self.weekly_remaining_capacity.get(wk, 0.0) - hours)

    def add_unavailable_period(self, start_date_str, end_date_str):
        try:
            start_dt = try_parse_date(start_date_str).replace(hour=0, minute=0, second=0)
            end_dt = try_parse_date(end_date_str).replace(hour=23, minute=59, second=59)
            if end_dt < start_dt:
                return
            self.unavailable_periods.append((start_dt, end_dt))
            self.unavailable_periods.sort()
        except ValueError:
            pass

    def _get_next_working_datetime(self, current_dt_orig):
        current_dt = current_dt_orig.replace(second=0, microsecond=0)
        max_iterations = 500
        iterations = 0
        while iterations < max_iterations:
            iterations += 1
            wd = current_dt.weekday()
            if wd >= 5:
                days_to_monday = 7 - wd
                current_dt = datetime.combine((current_dt + timedelta(days=days_to_monday)).date(), time.min)
                continue
            moved = False
            for un_start, un_end in self.unavailable_periods:
                if un_start <= current_dt <= un_end:
                    current_dt = (un_end + timedelta(minutes=1)).replace(second=0, microsecond=0)
                    moved = True
                    break
            if moved:
                continue
            return current_dt
        return current_dt_orig

    def calculate_end_datetime(self, start_dt_param, duration_hours):
        if duration_hours <= 0:
            return start_dt_param
        actual_start = self._get_next_working_datetime(start_dt_param)
        self._ensure_planning_start(actual_start)
        remaining_hours = duration_hours
        current_dt = actual_start
        max_weeks = 520
        for _ in range(max_weeks):
            if remaining_hours <= 0:
                break
            wk = self._relative_week_index(current_dt)
            if self.is_bottleneck and self._last_week_index is not None and wk > self._last_week_index:
                return datetime.max
            monday_of_wk = self.planning_start_monday + timedelta(days=7 * (wk - 1))
            week_end_dt = datetime.combine(monday_of_wk + timedelta(days=7), time.min)
            hours_to_week_end = self._working_hours_between(current_dt, week_end_dt)
            if hours_to_week_end <= 0:
                next_monday = week_end_dt
                next_working = self._get_next_working_datetime(next_monday)
                if next_working <= current_dt:
                    next_working = self._get_next_working_datetime(week_end_dt + timedelta(days=1))
                current_dt = next_working
                continue
            if remaining_hours <= hours_to_week_end:
                return self._advance_working_hours(current_dt, remaining_hours)
            else:
                remaining_hours -= hours_to_week_end
                current_dt = self._get_next_working_datetime(week_end_dt)
        return datetime.max

    def _working_hours_between(self, start_dt, end_dt):
        if end_dt <= start_dt:
            return 0.0
        total = 0.0
        current = start_dt
        while current < end_dt:
            wd = current.weekday()
            if wd >= 5:
                days_to_monday = 7 - wd
                current = datetime.combine((current + timedelta(days=days_to_monday)).date(), time.min)
                continue
            next_day = datetime.combine((current + timedelta(days=1)).date(), time.min)
            segment_end = min(end_dt, next_day)
            seg_hours = (segment_end - current).total_seconds() / 3600.0
            for un_start, un_end in self.unavailable_periods:
                overlap_start = max(current, un_start)
                overlap_end = min(segment_end, un_end)
                if overlap_end > overlap_start:
                    seg_hours -= (overlap_end - overlap_start).total_seconds() / 3600.0
            total += max(0.0, seg_hours)
            current = segment_end
        return total

    def _advance_working_hours(self, start_dt, hours):
        if hours <= 0:
            return start_dt
        remaining = hours
        current = start_dt
        max_iterations = 3650
        iteration = 0
        while remaining > 1e-9:
            iteration += 1
            if iteration > max_iterations:
                return current
            wd = current.weekday()
            if wd >= 5:
                days_to_monday = 7 - wd
                current = datetime.combine((current + timedelta(days=days_to_monday)).date(), time.min)
                continue
            in_unavail = False
            for un_start, un_end in self.unavailable_periods:
                if un_start <= current <= un_end:
                    current = un_end + timedelta(minutes=1)
                    in_unavail = True
                    break
            if in_unavail:
                continue
            next_day = datetime.combine((current + timedelta(days=1)).date(), time.min)
            segment_end = next_day
            for un_start, un_end in self.unavailable_periods:
                if un_start > current:
                    segment_end = min(segment_end, un_start)
            avail_hours = (segment_end - current).total_seconds() / 3600.0
            if avail_hours <= 0:
                current = segment_end + timedelta(minutes=1)
                continue
            if remaining <= avail_hours:
                result = current + timedelta(hours=remaining)
                return self._get_next_working_datetime(result)
            remaining -= avail_hours
            current = segment_end

    # =====================================================================
    # MODIFICATION : Capacité infinie pour les non‑goulot 
    # =====================================================================
    def _search_limit_datetime(self, search_start_dt_param):
        if not self.is_bottleneck:
            return datetime.max   # plus aucune limite temporelle
        if self._last_week_date is not None:
            return self._last_week_date
        return search_start_dt_param + timedelta(days=365)

    def find_available_slot(self, search_start_dt_param, duration_hours, of_id_to_ignore=None, reasons=None):
        if self.is_bottleneck and self._first_week_date is not None:
            if search_start_dt_param < self._first_week_date:
                if reasons is not None:
                    reasons.append(f"Recherche avant {self._first_week_date} -> aucun slot disponible")
                return None, None

        current_try_start_dt = self._get_next_working_datetime(search_start_dt_param)
        max_search_datetime = self._search_limit_datetime(search_start_dt_param)

        while current_try_start_dt < max_search_datetime:
            if self.is_bottleneck and self._last_week_date is not None:
                if current_try_start_dt >= self._last_week_date:
                    if reasons is not None:
                        reasons.append(f"Depassement de la derniere semaine de capacite ({self._last_week_date})")
                    break

            if not self._has_any_capacity_in_week(current_try_start_dt):
                if reasons is not None:
                    wk = self._relative_week_index(current_try_start_dt)
                    reasons.append(f"Semaine {wk} hors plage ou saturee")
                days_to_next_monday = (7 - current_try_start_dt.weekday()) % 7
                if days_to_next_monday == 0:
                    days_to_next_monday = 7
                next_week_monday = (current_try_start_dt + timedelta(days=days_to_next_monday)).replace(hour=0, minute=0)
                candidate = self._get_next_working_datetime(next_week_monday)
                if candidate <= current_try_start_dt:
                    candidate = self._get_next_working_datetime(next_week_monday + timedelta(days=1))
                current_try_start_dt = candidate
                continue

            potential_end_dt = self.calculate_end_datetime(current_try_start_dt, duration_hours)
            if potential_end_dt is None or potential_end_dt == datetime.max:
                if reasons is not None:
                    reasons.append(f"Impossible de calculer la fin a partir de {current_try_start_dt} (retourne None ou max)")
                current_try_start_dt = self._get_next_working_datetime(current_try_start_dt + timedelta(days=1))
                continue

            if not self._can_add_slot_in_week(current_try_start_dt, potential_end_dt):
                wk = self._relative_week_index(current_try_start_dt)
                allowed = self._allowed_hours_for_week(current_try_start_dt)
                used = self.weekly_load_hours[wk]
                dist = self._weekly_hours_distribution(current_try_start_dt, potential_end_dt)
                add_hours = dist.get(wk, 0)
                if reasons is not None:
                    reasons.append(f"Capacite insuffisante semaine {wk}: dispo {allowed-used:.2f}h, besoin {add_hours:.2f}h")
                days_to_next_monday = (7 - current_try_start_dt.weekday()) % 7
                if days_to_next_monday == 0:
                    days_to_next_monday = 7
                next_week_monday = (current_try_start_dt + timedelta(days=days_to_next_monday)).replace(hour=0, minute=0)
                candidate = self._get_next_working_datetime(next_week_monday)
                if candidate <= current_try_start_dt:
                    candidate = self._get_next_working_datetime(next_week_monday + timedelta(days=1))
                current_try_start_dt = candidate
                continue

            return current_try_start_dt, potential_end_dt

        return None, None

    def book_slot(self, start_dt, end_dt, of_id):
        self.clear_schedule_for_of(of_id)
        self.scheduled_slots.append((start_dt, end_dt, of_id))
        self._ensure_planning_start(start_dt)
        wk_s = self._relative_week_index(start_dt)
        wk_e = self._relative_week_index(end_dt)
        for wk in range(wk_s, wk_e + 1):
            self.slots_by_week[wk].append((start_dt, end_dt, of_id))
        self._register_slot_in_week_load(start_dt, end_dt)

    def clear_schedule_for_of(self, of_id):
        if not self.scheduled_slots:
            return
        to_remove = [(s, e, o) for (s, e, o) in self.scheduled_slots if o == of_id]
        if not to_remove:
            return
        self.scheduled_slots = [(s, e, o) for (s, e, o) in self.scheduled_slots if o != of_id]
        for s, e, _ in to_remove:
            self._ensure_planning_start(s)
            wk_s = self._relative_week_index(s)
            wk_e = self._relative_week_index(e)
            for wk in range(wk_s, wk_e + 1):
                bucket = self.slots_by_week.get(wk)
                if bucket:
                    self.slots_by_week[wk] = [(bs, be, bo) for bs, be, bo in bucket if bo != of_id]
            dist = self._weekly_hours_distribution(s, e)
            for wk, hours in dist.items():
                self.weekly_load_hours[wk] = max(0.0, self.weekly_load_hours[wk] - hours)
                if self.is_bottleneck:
                    cap = float(self.weekly_capacity_by_week.get(wk, 0.0))
                    self.weekly_remaining_capacity[wk] = min(cap, self.weekly_remaining_capacity.get(wk, 0.0) + hours)

    def __repr__(self):
        return (f"Post(id={self.id}, name='{self.name}', goulot={1 if self.is_bottleneck else 0}, "
                f"scheduled_slots={len(self.scheduled_slots)})")


from dataclasses import dataclass

@dataclass(slots=True)
class OperationAlternative:
    post_id: str
    standard_time_hours: float
    priority: int
    charge_origine: float = None   # ChargeCalculee avant evolution de quantite

@dataclass(slots=True)
class OperationStep:
    sequence: int
    operation_name: str
    alternatives: list


def op_charge_for_post(step, post_id):
    """ChargeCalculee (h) de l'operation sur le poste retenu (source : sim_of_work.ChargeCalculee)."""
    for alt in step.alternatives:
        if str(alt.post_id) == str(post_id):
            return float(alt.standard_time_hours or 0.0)
    return float(step.alternatives[0].standard_time_hours or 0.0) if step.alternatives else 0.0


def op_charge_origine_for_post(step, post_id):
    """ChargeCalculee (h) d'origine (avant evolution de quantite) sur le poste retenu."""
    def _orig(alt):
        v = alt.charge_origine if alt.charge_origine is not None else alt.standard_time_hours
        return float(v or 0.0)
    for alt in step.alternatives:
        if str(alt.post_id) == str(post_id):
            return _orig(alt)
    return _orig(step.alternatives[0]) if step.alternatives else 0.0


# ---------------------------------------------------------------------
# EVOLUTION DE LA QUANTITE A FABRIQUER PAR ARTICLE
# ---------------------------------------------------------------------
def load_evol_qte_map(spark=None, table_name=None):
    """Retourne {article_normalise: (article_brut, pct)} a partir de EVOL_QTE_PAR_ARTICLE
    et, si renseignee, de la table EVOL_QTE_TABLE (colonnes Article, Pct)."""
    evol = {}
    if table_name and spark is not None:
        try:
            for r in spark.table(table_name).select("Article", "Pct").collect():
                if r["Article"] and r["Pct"] is not None:
                    evol[_norm(r["Article"])] = (str(r["Article"]).strip(), float(r["Pct"]))
            print(f"[Evol] {len(evol)} articles lus depuis {table_name}")
        except Exception as e:
            raise RuntimeError(f"[Evol] Lecture impossible de la table {table_name} : {e}")
    for art, pct in (EVOL_QTE_PAR_ARTICLE or {}).items():
        if art and pct is not None:
            evol[_norm(art)] = (str(art).strip(), float(pct))
    return evol


def describe_evol_qte(evol_map):
    """Texte court pour la table Simulation, ex 'PF001234:+10; PF005678:-5'."""
    if not evol_map:
        return "Aucune"
    parts = [f"{raw}:{pct:+g}" for raw, pct in sorted(evol_map.values())]
    txt = "; ".join(parts)
    if EVOL_QTE_PROPAGER_SF:
        txt += " (SF propages)"
    return txt if len(txt) <= 4000 else txt[:3990] + " ..."


def _bom_topological_order(bom_data):
    """Produits normalises ordonnes parents avant enfants (Kahn). Les cycles eventuels
    sont ajoutes a la fin pour ne jamais bloquer."""
    children = defaultdict(set)
    indeg = defaultdict(int)
    nodes = set()
    for b in bom_data:
        p, c = b.parent_product_id, b.child_product_id
        if not p or not c or p == c:
            continue
        nodes.update((p, c))
        if c not in children[p]:
            children[p].add(c)
            indeg[c] += 1
    queue = deque(sorted(n for n in nodes if indeg[n] == 0))
    order = []
    while queue:
        n = queue.popleft()
        order.append(n)
        for c in sorted(children.get(n, ())):
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)
    if len(order) < len(nodes):
        reste = sorted(nodes - set(order))
        print(f"[Evol][WARN] Cycle detecte dans la nomenclature ({len(reste)} articles) : ordre arbitraire pour ces articles")
        order.extend(reste)
    return order


def apply_quantity_evolution(all_ofs, operations_map, bom_data, evol_map):
    """Applique l'evolution de quantite par article :
      1) articles listes : quantite x (1 + pct/100) ;
      2) si EVOL_QTE_PROPAGER_SF : chaque composant recoit le surplus (ou la baisse) reellement
         consomme par ses parents via la nomenclature, reparti au prorata sur ses OF de l'horizon
         (ex : surplus 60 pour 1000 en OF -> +6 %). Un article liste explicitement garde son %.
      3) les heures des operations de chaque OF suivent le meme rapport que la quantite.
    Retourne {article_normalise: coef} pour les articles modifies."""
    import math

    for of in all_ofs:
        of.qty_origine = of.quantity
        of.coef_evol = 1.0
    if not evol_map:
        print("[Evol] Aucune evolution de quantite (EVOL_QTE_PAR_ARTICLE vide)")
        return {}

    ofs_by_product = defaultdict(list)
    for of in all_ofs:
        ofs_by_product[of.product_id].append(of)

    inconnus = [raw for norm, (raw, _) in evol_map.items() if norm not in ofs_by_product]
    if inconnus:
        print(f"[Evol][WARN] {len(inconnus)} article(s) sans OF dans l'horizon, ignore(s) : {inconnus[:20]}")

    coef_by_product = {norm: max(0.0, 1.0 + pct / 100.0)
                       for norm, (raw, pct) in evol_map.items() if norm in ofs_by_product}
    explicit = set(coef_by_product)

    if EVOL_QTE_PROPAGER_SF:
        bom_children = defaultdict(list)
        for b in bom_data:
            if b.parent_product_id and b.child_product_id and b.parent_product_id != b.child_product_id:
                bom_children[b.parent_product_id].append((b.child_product_id, b.quantity_child_per_parent))
        extra_need = defaultdict(float)
        for pid in _bom_topological_order(bom_data):
            ofs_p = ofs_by_product.get(pid, [])
            total_p = sum(o.qty_origine for o in ofs_p)
            if pid not in explicit and pid in extra_need and total_p > 0:
                coef_by_product[pid] = max(0.0, 1.0 + extra_need[pid] / total_p)
            coef_p = coef_by_product.get(pid, 1.0)
            if coef_p == 1.0 or total_p <= 0:
                continue
            extra_p = total_p * (coef_p - 1.0)
            for child, q in bom_children.get(pid, []):
                extra_need[child] += extra_p * q
        sans_of = [c for c in extra_need if c not in ofs_by_product and abs(extra_need[c]) > 1e-9]
        if sans_of:
            print(f"[Evol][INFO] {len(sans_of)} composant(s) impacte(s) sans OF dans l'horizon (non modifies)")

    nb_of = 0
    for pid, coef in coef_by_product.items():
        if abs(coef - 1.0) < 1e-12:
            continue
        for of in ofs_by_product.get(pid, []):
            new_qty = of.qty_origine * coef
            if EVOL_QTE_ARRONDI:
                new_qty = float(math.ceil(new_qty - 1e-9))
            of.quantity = new_qty
            of.coef_evol = coef
            of.source_qty = f"{new_qty:g}" if float(new_qty).is_integer() else f"{new_qty:.3f}"
            ratio = (new_qty / of.qty_origine) if of.qty_origine else coef
            for step in operations_map.get(of.normalized_id, []):
                for alt in step.alternatives:
                    if alt.charge_origine is None:
                        alt.charge_origine = alt.standard_time_hours
                    alt.standard_time_hours = float(alt.charge_origine or 0.0) * ratio
            nb_of += 1

    for norm in sorted(explicit):
        ofs_p = ofs_by_product[norm]
        avant = sum(o.qty_origine for o in ofs_p)
        apres = sum(o.quantity for o in ofs_p)
        print(f"[Evol] {evol_map[norm][0]} : {evol_map[norm][1]:+g} % -> {len(ofs_p)} OF, "
              f"qte {avant:g} -> {apres:g}")
    propages = {p: c for p, c in coef_by_product.items() if p not in explicit and abs(c - 1.0) >= 1e-12}
    if propages:
        print(f"[Evol] {len(propages)} composant(s) ajuste(s) par propagation nomenclature, ex : "
              + ", ".join(f"{ofs_by_product[p][0].raw_product_id} x{c:.3f}"
                          for p, c in sorted(propages.items())[:10]))
    print(f"[Evol] Total : {nb_of} OF modifies (quantite et charge des operations)")
    return {p: c for p, c in coef_by_product.items() if abs(c - 1.0) >= 1e-12}


# ---------------------------------------------------------------------
# ALGORITHME DE GROUPAGE
# ---------------------------------------------------------------------
class _BomGraph(defaultdict):
    pass


def build_bom_graph(bom_data):
    G = _BomGraph(set)
    for b in bom_data:
        p, c = b.parent_product_id, b.child_product_id
        if p and c:
            G[p].add(c)
            G[c].add(p)

    parents_of = defaultdict(set)
    for b in bom_data:
        if b.parent_product_id and b.child_product_id:
            parents_of[b.child_product_id].add(b.parent_product_id)

    sf_sf_hubs = {
        child: parents for child, parents in parents_of.items()
        if len(parents) >= 2 and all(p.startswith('SF') for p in parents)
    }
    G.sf_sf_hubs = sf_sf_hubs
    return G


def connected_component_nodes(G, start):
    start_norm = _norm(start)
    if not start_norm:
        return set()
    sf_sf_hubs = getattr(G, 'sf_sf_hubs', {})
    seen = {start_norm}
    q = deque([start_norm])
    while q:
        u = q.popleft()
        for v in G.get(u, ()):
            if v in seen:
                continue
            if u in sf_sf_hubs and v in sf_sf_hubs[u] and start_norm in sf_sf_hubs[u] and start_norm != v:
                continue
            seen.add(v)
            q.append(v)
    return seen


def _get_monday(dt):
    return dt - timedelta(days=dt.weekday())


def run_grouping_algorithm(all_ofs, bom_data, horizon_H_days_param):
    product_level = {}
    for b in bom_data:
        p, c, cl, pl = b.parent_product_id, b.child_product_id, b.child_bom_level, b.parent_bom_level
        if c and cl > 0:
            product_level[c] = max(product_level.get(c, 0), cl)
        if p and pl > 0:
            product_level[p] = max(product_level.get(p, 0), pl)

    for of in all_ofs:
        of.bom_level = product_level.get(of.product_id, 0)

    def get_level(of):
        return of.bom_level

    bom_graph = build_bom_graph(bom_data)
    ofs_by_id = {of.id: of for of in all_ofs}
    ofs_by_product = defaultdict(list)
    for of in all_ofs:
        ofs_by_product[of.product_id].append(of)

    # ---- Exclure les OFs de catégorie "118" du groupement ----
    unassigned_ids = {
        of.id for of in all_ofs
        if of.assigned_group_id is None
        and not (hasattr(of, 'category') and of.category == "118")
    }
    group_counter = 1
    groups = []
    skipped = set()

    while unassigned_ids:
        best_of, best_level, best_date = None, -1, None
        for of_id in unassigned_ids:
            if of_id in skipped:
                continue
            of = ofs_by_id[of_id]
            lvl = get_level(of)
            if best_of is None or lvl > best_level or (lvl == best_level and of.need_date < best_date) or \
               (lvl == best_level and of.need_date == best_date and of.id < best_of.id):
                best_of, best_level, best_date = of, lvl, of.need_date
        anchor_of = best_of
        if anchor_of is None:
            break

        # ----- Le bloc spécifique "category == 118" est supprimé -----

        anchor_pid_norm = anchor_of.product_id
        family = connected_component_nodes(bom_graph, anchor_of.product_id)
        if not family:
            family = {anchor_pid_norm}

        h_days = horizon_H_days_param
        if anchor_of.horizon_days not in (None, ""):
            try:
                h_days = max(0, int(anchor_of.horizon_days))
            except Exception:
                h_days = horizon_H_days_param

        window_start = anchor_of.need_date
        window_end = window_start + timedelta(days=h_days)

        raw_candidates = []
        for pid in family:
            for of in ofs_by_product.get(pid, []):
                # Exclure les OFs de catégorie "118" des candidats
                if hasattr(of, 'category') and of.category == "118":
                    continue
                if of.id in unassigned_ids and window_start <= of.need_date <= window_end:
                    raw_candidates.append(of)

        cand_pids_raw = {of.product_id for of in raw_candidates}
        related_pids = set()
        for b in bom_data:
            p, c = b.parent_product_id, b.child_product_id
            if p in cand_pids_raw and c in cand_pids_raw:
                related_pids.add(p)
                related_pids.add(c)

        if anchor_of.product_type == "SF":
            pid_counts = defaultdict(int)
            for of in raw_candidates:
                pid_counts[of.product_id] += 1
            for pid, cnt in pid_counts.items():
                if cnt > 1:
                    related_pids.add(pid)
        related_pids.add(anchor_pid_norm)

        candidates = [of for of in raw_candidates if of.product_id in related_pids]

        if len(candidates) <= 1 or anchor_of not in candidates:
            if anchor_of.product_type == "SF":
                current_group = Group(
                    id=str(group_counter),
                    ps_product_id=anchor_of.raw_product_id,
                    initial_ps_of=anchor_of,
                    window_start_date=anchor_of.need_date,
                    window_end_date=anchor_of.need_date + timedelta(days=h_days),
                    initial_ps_as_stock=False,
                )
                unassigned_ids.discard(anchor_of.id)
                current_group.calculate_consumption(bom_data)
                groups.append(current_group)
                group_counter += 1
            else:
                skipped.add(anchor_of.id)
            continue

        if anchor_of.product_type != "SF":
            distinct_bom_linked_pids = set()
            cand_pids = {o.product_id for o in candidates}
            for b in bom_data:
                p, c = b.parent_product_id, b.child_product_id
                if p in cand_pids and c in cand_pids and p != c:
                    distinct_bom_linked_pids.add(p)
                    distinct_bom_linked_pids.add(c)
            if len(distinct_bom_linked_pids) < 2:
                skipped.add(anchor_of.id)
                continue

        current_group = Group(
            id=str(group_counter),
            ps_product_id=anchor_of.raw_product_id,
            initial_ps_of=anchor_of,
            window_start_date=window_start,
            window_end_date=window_end,
            initial_ps_as_stock=False,
        )

        direct_sf_children = defaultdict(set)
        for b in bom_data:
            p, c = b.parent_product_id, b.child_product_id
            if p and c and c.startswith("SF"):
                direct_sf_children[p].add(c)

        candidates_sorted = sorted(candidates, key=lambda o: (-get_level(o), o.need_date, o.product_id))
        for ofx in candidates_sorted:
            if ofx.id == anchor_of.id:
                continue
            if ofx.product_type == "PF":
                sf_children = direct_sf_children.get(ofx.product_id, set())
                if sf_children:
                    group_pids = {o.product_id for o in current_group.ofs}
                    if not sf_children.intersection(group_pids):
                        continue
            current_group.add_of(ofx, ps_quantity_change=0)
            unassigned_ids.discard(ofx.id)

        unassigned_ids.discard(anchor_of.id)
        current_group.calculate_consumption(bom_data)
        groups.append(current_group)
        group_counter += 1

    for group in groups:
        anchor_monday = _get_monday(group.ofs[0].need_date)
        for of in group.ofs:
            of.need_date = anchor_monday

    for of in all_ofs:
        of.need_date = _get_monday(of.need_date)

    return groups, all_ofs


# ---------------------------------------------------------------------
# LISSAGE ET PLANIFICATION
# ---------------------------------------------------------------------
def smooth_and_schedule_groups(groups, all_ofs_with_groups, bom_data, posts_map, operations_map, params):
    import heapq
    import os
    from datetime import datetime
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType

    # Récupération des flags de simulation
    ignore_gap = params.get("ignore_gap", False)
    ignore_stock = params.get("ignore_stock", False)
    ignore_missing_posts = params.get("ignore_missing_posts", False)
    ignore_window = params.get("ignore_window", False)  # ignore aussi la fenetre avance/retard par article

    max_gap_hours = params.get("max_gap_between_ops_hours")
    if max_gap_hours is not None:
        max_gap_delta = timedelta(hours=max_gap_hours)
    else:
        max_gap_delta = timedelta(weeks=params.get("max_gap_between_ops_weeks", 2))
    max_gap_seconds = max_gap_delta.total_seconds()

    # Si ignore_gap est True, on met une valeur énorme pour ne jamais bloquer
    if ignore_gap:
        max_gap_seconds = 10**18

    week_sort_order = params.get('week_sort_order', 'earliest')

    # ---- Activation des logs ----
    enable_logging = (LOG_LEVEL == "detailed" and LOG_FILTER != "none")
    failure_records = [] if enable_logging else None

    # ---- Classification des raisons d'échec ----
    def classify_failure(reason: str) -> tuple:
        if not reason:
            return ("Autre", "Autre raison")
        r = reason.lower()
        if "stock insuffisant" in r or "stock" in r:
            return ("Stock", "Stock composant")
        if "capacite" in r or "capacité" in r:
            return ("Capacité", "Capacité insuffisante")
        if "poste inexistant" in r:
            return ("Poste inexistant", "Poste absent de SIM_POSTE_CAPA")
        if "ecart trop grand" in r:
            return ("Ecart", "Ecart entre opérations > MAX_GAP")
        if "fenetre depassee" in r or "impossible de calculer la fin" in r or "fenêtre" in r:
            return ("Fenêtre temporelle", "Poste indisponible (calcul fin)")
        if "chevauchement" in r:
            return ("Chevauchement", "Chevauchement OF")
        if "semaine" in r and ("hors plage" in r or "saturee" in r):
            return ("Capacité", "Capacité insuffisante")
        return ("Autre", "Autre raison")

    def log_failure(of_id, step_name, post_id, reason, start_attempt=None, end_attempt=None,
                    week_start=None, division=None, num_plan=None):
        if not enable_logging:
            return
        if LOG_FILTER != "all" and division != LOG_FILTER:
            return
        typologie, categorie = classify_failure(reason)
        composant = post_id
        if typologie == "Stock" and "Stock insuffisant pour" in reason:
            parts = reason.split("pour")
            if len(parts) > 1:
                comp = parts[1].strip().split()[0] if parts[1].strip() else None
                if comp:
                    composant = comp
        elif post_id and post_id != "N/A":
            composant = post_id

        record = {
            "OF_Numero": of_id,
            "Operation": step_name,
            "Poste": post_id,
            "Typologie_Echec": typologie,
            "Categorie_Raison": categorie,
            "Date_debut_tentative": start_attempt,
            "Date_fin_tentative": end_attempt,
            "Semaine_candidate": week_start,
            "Statut_final": "ECHOUE",
            "Division": division or "",
            "Composant_ou_poste_concerne": composant,
            "Nb_tentatives": 1,
            "Num_plan": num_plan
        }
        if params.get("id_sim") is not None:
            record["ID_SIM"] = params["id_sim"]
        failure_records.append(record)

    # -------------------------------------------------------------

    def dt_to_str(dt):
        return dt.strftime("%Y-%m-%d %H:%M") if dt else None

    def days_delay_if_late(scheduled_end_dt, need_dt):
        if scheduled_end_dt is None:
            return 0
        delta = (scheduled_end_dt.date() - need_dt.date()).days
        return delta if delta > 0 else 0

    def qty_besoin_raw(of_obj):
        raw = getattr(of_obj, "source_qty", None)
        raw = (str(raw).strip() if raw is not None else "")
        if raw:
            return raw
        try:
            return str(float(getattr(of_obj, "quantity", 0) or 0))
        except Exception:
            return ""

    default_advance_days = int(params.get("advance_retreat_days", ADVANCE_RETREAT_DAYS))
    default_retard_days = int(params.get("advance_retreat_days", ADVANCE_RETREAT_DAYS))

    # Plancher "pas de passe" : lundi de la semaine de lancement de la simulation.
    # Aucune semaine candidate ne doit jamais etre anterieure a cette date,
    # meme si l'avance (advance_days) calculee depuis la need_date le suggererait.
    min_week_monday = params.get("min_week_monday", SIM_LAUNCH_MONDAY)

    # NB: get_advance_days/get_retard_days ne sont utilisees que quand
    # ignore_window est False (cf. _schedule_of_core) : en capacite infinie,
    # la fenetre article (CST1_0/CST2_0) est completement court-circuitee
    # par datetime.max, sans aucun plafond en jours.
    def get_advance_days(of_obj):
        v = getattr(of_obj, "advance_days", None)
        if v in (None, "", 0):
            return default_advance_days
        try:
            return max(0, int(v))
        except Exception:
            return default_advance_days

    def get_retard_days(of_obj):
        v = getattr(of_obj, "retard_days", None)
        if v in (None, "", 0):
            return default_retard_days
        try:
            return max(0, int(v))
        except Exception:
            return default_retard_days

    def candidate_week_starts(need_dt, advance_days, retard_days, sort_order='earliest'):
        need_monday = need_dt - timedelta(days=need_dt.weekday())
        earliest = need_monday - timedelta(days=advance_days)
        latest = need_monday + timedelta(days=retard_days)
        earliest_monday = earliest - timedelta(days=earliest.weekday())

        # --- Controle "pas de passe" ---------------------------------
        # On ne planifie jamais une semaine avant le lundi de lancement
        # de la simulation (min_week_monday), quel que soit MODE_PLANIF.
        if min_week_monday is not None and earliest_monday < min_week_monday:
            earliest_monday = min_week_monday
        # Si meme la fenetre de retard reste entierement dans le passe
        # (OF deja tres en retard), on ramene aussi "latest" au lundi de
        # lancement pour que l'OF reste planifiable des la semaine en cours
        # plutot que d'echouer faute de semaine candidate.
        if min_week_monday is not None and latest < min_week_monday:
            latest = min_week_monday
        # ---------------------------------------------------------------

        weeks = []
        cur = earliest_monday
        while cur <= latest:
            weeks.append(cur)
            cur += timedelta(weeks=1)
        if sort_order == 'closest':
            def sort_key(monday):
                diff_days = (monday - need_monday).days
                if diff_days == 0:
                    return (0, 0)
                elif diff_days < 0:
                    return (1, -diff_days)
                else:
                    return (2, diff_days)
            weeks.sort(key=sort_key)
        return weeks

    product_level = {}
    for b in bom_data:
        p, c = b.parent_product_id, b.child_product_id
        if c:
            product_level[c] = max(product_level.get(c, 0), b.child_bom_level or 0)
        if p:
            product_level[p] = max(product_level.get(p, 0), b.parent_bom_level or 0)

    bom_children = defaultdict(list)
    for b in bom_data:
        bom_children[b.parent_product_id].append((b.child_product_id, b.quantity_child_per_parent))

    global_lots = defaultdict(list)

    # Tracabilite du stock SF : une ligne par entree (lot produit) et par sortie (lot consomme)
    stock_movements = []
    raw_by_norm = {}
    for _o in all_ofs_with_groups:
        raw_by_norm.setdefault(_o.product_id, _o.raw_product_id)

    def check_component_availability(prod_norm, qty_parent, ignore_of_id=None):
        if ignore_stock:
            return True, None, {}, ""
        children = bom_children.get(prod_norm, [])
        if not children:
            return True, None, {}, ""
        allocations = {}
        latest_ready = None
        for child_norm, coef in children:
            needed = qty_parent * coef
            if needed <= 0:
                continue
            heap = global_lots.get(child_norm, [])
            if not heap:
                return False, None, {}, f"Stock insuffisant pour {child_norm}"
            tmp_heap = heap[:]
            heapq.heapify(tmp_heap)
            taken = []
            remaining = needed
            while remaining > 0 and tmp_heap:
                end_dt, qty_free, of_id = heapq.heappop(tmp_heap)
                if ignore_of_id and of_id == ignore_of_id:
                    continue
                take = min(qty_free, remaining)
                if take > 0:
                    taken.append((end_dt, take, of_id))
                    remaining -= take
                    if remaining <= 1e-9:
                        if latest_ready is None or end_dt > latest_ready:
                            latest_ready = end_dt
                        break
            if remaining > 1e-9:
                return False, None, {}, f"Stock insuffisant pour {child_norm}"
            allocations[child_norm] = taken
        return True, latest_ready, allocations, ""

    def commit_allocations(allocations, consumer_of_id=None, move_dt=None):
        if ignore_stock:
            return
        for child_norm, taken_list in allocations.items():
            # Trace des sorties : un mouvement par lot consomme
            for t_end_dt, t_qty, t_of_id in taken_list:
                stock_movements.append({
                    "article": raw_by_norm.get(child_norm, child_norm),
                    "of_id": consumer_of_id,
                    "mouvement": "sortie",
                    "qty": -float(t_qty),
                    "date": move_dt,
                    "of_source": t_of_id,
                })
            heap = global_lots.get(child_norm, [])
            new_heap = []
            for end_dt, qty_free, of_id in heap:
                used = 0
                for t_end_dt, t_qty, t_of_id in taken_list:
                    if t_of_id == of_id and abs((t_end_dt - end_dt).total_seconds()) < 1:
                        used += t_qty
                qty_free -= used
                if qty_free > 0:
                    new_heap.append((end_dt, qty_free, of_id))
            heapq.heapify(new_heap)
            global_lots[child_norm] = new_heap

    def add_produced_lot(prod_norm, end_dt, qty, of_id):
        if ignore_stock:
            return
        heapq.heappush(global_lots[prod_norm], (end_dt, qty, of_id))
        stock_movements.append({
            "article": raw_by_norm.get(prod_norm, prod_norm),
            "of_id": of_id,
            "mouvement": "entree",
            "qty": float(qty),
            "date": end_dt,
            "of_source": of_id,
        })

    _next_working_cache = {}

    def get_next_working_datetime_cached(post, dt):
        key = (post.id, dt.year, dt.month, dt.day, dt.hour)
        if key in _next_working_cache:
            return _next_working_cache[key]
        res = post._get_next_working_datetime(dt)
        if len(_next_working_cache) < 10000:
            _next_working_cache[key] = res
        return res

    def get_operations(of_obj):
        key_of = of_obj.normalized_id
        key_prod = of_obj.product_id
        key_type = of_obj.product_type.upper()
        return (operations_map.get(key_of) or operations_map.get(key_prod) or operations_map.get(key_type, []))

    def clear_of_schedules(of_id, ops_steps):
        for step in ops_steps:
            for alt in step.alternatives:
                post = posts_map.get(alt.post_id)
                if post:
                    post.clear_schedule_for_of(of_id + "_" + step.operation_name)

    # -------------------------------------------------------------
    # try_schedule_step retourne (s_dt, e_dt, None) ou (None, None, raison)
    # -------------------------------------------------------------
    def try_schedule_step(step, alt, post, start_search, chain_last_end, max_gap_seconds,
                           latest_allowed, of_id, division, need_date, week_start, num_plan, group_context=False, _dbg=False):
        candidate_start = start_search
        if chain_last_end is not None:
            candidate_start = max(candidate_start, chain_last_end)
        candidate_start = get_next_working_datetime_cached(post, candidate_start)

        reasons = [] if _dbg else None
        s_dt, e_dt = post.find_available_slot(candidate_start, alt.standard_time_hours,
                                               of_id + "_" + step.operation_name, reasons=reasons)
        if not s_dt or not e_dt:
            if enable_logging:
                if reasons:
                    reason_str = "; ".join(reasons)
                    log_failure(of_id, step.operation_name, post.id, reason_str, candidate_start, None, week_start, division, num_plan)
                else:
                    log_failure(of_id, step.operation_name, post.id, "Slot non trouvé (sans raison)", candidate_start, None, week_start, division, num_plan)
            return None, None, "slot_introuvable"
        if chain_last_end is not None and (s_dt - chain_last_end).total_seconds() > max_gap_seconds:
            if enable_logging:
                reason = f"Ecart trop grand entre opérations ({int((s_dt - chain_last_end).total_seconds()/3600)}h > {int(max_gap_seconds/3600)}h)"
                log_failure(of_id, step.operation_name, post.id, reason, s_dt, e_dt, week_start, division, num_plan)
            # Si ignore_gap est True, on ne retourne pas None, on continue
            if not ignore_gap:
                return None, None, "ecart_trop_grand"
        if e_dt > latest_allowed:
            if enable_logging:
                reason = f"Fin de l'opération ({e_dt}) après la limite autorisée ({latest_allowed})"
                log_failure(of_id, step.operation_name, post.id, reason, s_dt, e_dt, week_start, division, num_plan)
            return None, None, "depassement_fenetre"
        return s_dt, e_dt, None

    # -------------------------------------------------------------
    # select_best_alternative retourne (alt, post, s, e, None) ou (None,None,None,None, raison)
    # -------------------------------------------------------------
    def select_best_alternative(step, context):
        missing_post_detected = False
        for alt in step.alternatives:
            post = posts_map.get(alt.post_id)
            if not post:
                missing_post_detected = True
                if enable_logging:
                    log_failure(
                        context['of_id'],
                        step.operation_name,
                        alt.post_id,
                        "Poste inexistant dans la base de capacité",
                        context['start_search'],
                        None,
                        context['week_start'],
                        context['division'],
                        context['num_plan']
                    )
                continue
            result = try_schedule_step(
                step, alt, post,
                start_search=context['start_search'],
                chain_last_end=context['chain_last_end'],
                max_gap_seconds=context['max_gap_seconds'],
                latest_allowed=context['latest_allowed'],
                of_id=context['of_id'],
                division=context['division'],
                need_date=context['need_date'],
                week_start=context['week_start'],
                num_plan=context['num_plan'],
                group_context=False,
                _dbg=enable_logging
            )
            if result[0] is not None:
                return alt, post, result[0], result[1], None
        # Aucune alternative n'a fonctionné
        if missing_post_detected:
            return None, None, None, None, "poste_absent"
        else:
            return None, None, None, None, "capacite_insuffisante"

    smoothing_items = []
    scheduled_ofs = []
    _plan_counter = [0]

    # --- Helper pour le group_id final (catégorie 118 → "INDIVIDUEL") ---
    def get_display_group_id(of_obj, group_id_str):
        if getattr(of_obj, "category", "") == "118":
            return "INDIVIDUEL"
        else:
            return group_id_str if group_id_str is not None else "INDIVIDUEL"

    def _make_failure_item(of_obj, group_id_str, status, debug_msg):
        group_id_final = get_display_group_id(of_obj, group_id_str)
        return {
            "of_id": of_obj.id, "product_id": of_obj.raw_product_id, "designation": of_obj.designation,
            "group_id": group_id_final, "qty_besoin": qty_besoin_raw(of_obj),
            "date_x3": of_obj.original_need_date.strftime("%Y-%m-%d"),   # originale avant need_date
            "need_date": of_obj.need_date.strftime("%Y-%m-%d"),
            "scheduled_start": None, "scheduled_end": None, "status": status,
            "retard_jours": 0, "avance_jours": 0, "operations": [],
            "num_plan": of_obj.num_plan, "priority": of_obj.priority, "division": of_obj.division or "",
            "horizon": of_obj.horizon_days,
            "category": of_obj.category or ""
        }
    #
    def _schedule_of_core(of_to_schedule, group_id_str, group_context=False, gp_last_end=None, tmp_post_end=None):
        _plan_counter[0] += 1
        of_to_schedule.num_plan = _plan_counter[0]

        need_dt = of_to_schedule.need_date
        ops = get_operations(of_to_schedule)   # <-- DEFINED EARLY

        # ---- SIMULATION INFINIE : on place l'OF à sa date de besoin, mais on
        # planifie quand meme les operations sur les postes (sans contrainte
        # de capacite, puisque force_infinite_capacity a deja mis tous les
        # postes en is_bottleneck=False) afin que weekly_load_hours (et donc
        # PlannedHours dans S_weekly_capacity) refletent la charge reelle. ----
        if ignore_window:  # Infinie simple (VERIF_FENETRE=False)
            # --- Verifications optionnelles : stock et ecart (memes regles et statuts qu'en mode fini) ---
            # Chaque fonction/flag est neutre quand la verification est desactivee.
            stock_ok, components_ready_dt, allocations, stock_reason = check_component_availability(
                of_to_schedule.product_id, of_to_schedule.quantity, ignore_of_id=of_to_schedule.id
            )
            if not stock_ok:
                status = "ECHOUE(stock insuffisant)"
                of_to_schedule.status = status
                of_to_schedule.scheduled_start_date = None
                of_to_schedule.scheduled_end_date = None
                smoothing_items.append(_make_failure_item(of_to_schedule, group_id_str, status, stock_reason))
                scheduled_ofs.append(of_to_schedule)
                if enable_logging:
                    log_failure(of_to_schedule.id, "STOCK", "", stock_reason, None, None, None,
                                of_to_schedule.division, of_to_schedule.num_plan)
                return of_to_schedule

            gap_failure = None
            ops_chain = []
            missing_post_ops = []
            chain_last_end = None
            # Plancher "pas de passe" : meme en capacite infinie, un OF n'est jamais
            # place avant le lundi de lancement (SIM_LAUNCH_DATE).
            plan_dt = max(need_dt, min_week_monday) if min_week_monday is not None else need_dt
            floor_dt = plan_dt
            # Stock : l'OF ne peut pas demarrer avant que les composants soient disponibles
            if components_ready_dt is not None:
                plan_dt = max(plan_dt, components_ready_dt)
            # Arrets de la simulation (SIM_ARRETS) : meme en capacite infinie, l'OF n'est pas
            # place pendant un arret. Il est repousse a la fin de l'arret (compte comme retard,
            # car floor_dt reste la date de besoin).
            plan_dt = push_out_of_arrets(plan_dt)
            start_search = plan_dt
            for step in ops:
                chosen_post = None
                s_dt = e_dt = None
                step_gap_hours = None
                for alt in step.alternatives:
                    cand_post = posts_map.get(alt.post_id)
                    if cand_post is None:
                        if enable_logging:
                            log_failure(
                                of_to_schedule.id,
                                step.operation_name,
                                alt.post_id,
                                "Poste inexistant dans la base de capacité",
                                chain_last_end if chain_last_end is not None else start_search,
                                None,
                                None,
                                of_to_schedule.division,
                                of_to_schedule.num_plan
                            )
                        continue
                    candidate_start = chain_last_end if chain_last_end is not None else start_search
                    candidate_start = get_next_working_datetime_cached(cand_post, candidate_start)
                    try_s_dt, try_e_dt = cand_post.find_available_slot(
                        candidate_start, alt.standard_time_hours,
                        of_to_schedule.id + "_" + step.operation_name
                    )
                    if try_s_dt is not None:
                        # Ecart max entre operations (VERIF_ECART_OPS)
                        if (not ignore_gap) and chain_last_end is not None and \
                                (try_s_dt - chain_last_end).total_seconds() > max_gap_seconds:
                            step_gap_hours = int((try_s_dt - chain_last_end).total_seconds() / 3600)
                            continue
                        chosen_post, s_dt, e_dt = cand_post, try_s_dt, try_e_dt
                        break
                if chosen_post is not None:
                    ops_chain.append((step, chosen_post, s_dt, e_dt))
                    chain_last_end = e_dt
                elif step_gap_hours is not None:
                    gap_failure = (step, step_gap_hours)
                    break
                else:
                    # aucune alternative n'a fonctionne (poste manquant pour toutes
                    # les alternatives de cette etape) -> on le trace explicitement
                    missing_post_ops.append(step)

            if gap_failure is not None:
                status = "ECHOUE(ecart trop grand)"
                of_to_schedule.status = status
                of_to_schedule.scheduled_start_date = None
                of_to_schedule.scheduled_end_date = None
                smoothing_items.append(_make_failure_item(
                    of_to_schedule, group_id_str, status,
                    f"Ecart trop grand ({gap_failure[1]}h > {int(max_gap_seconds/3600)}h) avant {gap_failure[0].operation_name}"
                ))
                scheduled_ofs.append(of_to_schedule)
                if enable_logging:
                    log_failure(of_to_schedule.id, gap_failure[0].operation_name, "",
                                f"Ecart trop grand entre operations ({gap_failure[1]}h > {int(max_gap_seconds/3600)}h)",
                                None, None, None, of_to_schedule.division, of_to_schedule.num_plan)
                return of_to_schedule

            for step, post, s_dt, e_dt in ops_chain:
                post.book_slot(s_dt, e_dt, of_to_schedule.id + "_" + step.operation_name)

            of_to_schedule.scheduled_start_date = plan_dt
            of_to_schedule.scheduled_end_date = plan_dt
            # Stock : consommation des composants + declaration du lot produit (neutres si stock ignore)
            commit_allocations(allocations, of_to_schedule.id, plan_dt)
            if of_to_schedule.quantity > 0:
                _lot_end = ops_chain[-1][3] if ops_chain else plan_dt
                add_produced_lot(of_to_schedule.product_id, _lot_end, of_to_schedule.quantity, of_to_schedule.id)
            # Retard uniquement si c'est le stock qui a decale l'OF apres sa date de besoin
            # NON uniquement si le stock decale l'OF d'au moins 1 jour calendaire
            # (comparaison sur les dates, sans l'heure : un decalage de quelques heures reste OUI)
            _retard = days_delay_if_late(plan_dt, floor_dt)
            _stock_late = _retard > 0
            of_to_schedule.status = "NON" if _stock_late else "OUI"
            group_id_final = get_display_group_id(of_to_schedule, group_id_str)
            smoothing_items.append({
                "of_id": of_to_schedule.id,
                "product_id": of_to_schedule.raw_product_id,
                "designation": of_to_schedule.designation,
                "group_id": group_id_final,
                "qty_besoin": qty_besoin_raw(of_to_schedule),
                "date_x3": of_to_schedule.original_need_date.strftime("%Y-%m-%d"),
                "need_date": need_dt.strftime("%Y-%m-%d"),
                "scheduled_start": plan_dt.strftime("%Y-%m-%d %H:%M"),
                "scheduled_end": plan_dt.strftime("%Y-%m-%d %H:%M"),
                "status": of_to_schedule.status,
                "retard_jours": _retard,
                "avance_jours": 0,
                "num_plan": of_to_schedule.num_plan,
                "priority": of_to_schedule.priority,
                "operations": [
                    {"operation": step.operation_name, "post_id": post.id, "start": dt_to_str(s_dt), "end": dt_to_str(e_dt),
                     "charge": op_charge_for_post(step, post.id),
                     "charge_origine": op_charge_origine_for_post(step, post.id)}
                    for step, post, s_dt, e_dt in ops_chain
                ] + [
                    {"operation": step.operation_name, "post_id": "poste inexistant", "start": None, "end": None, "charge": None, "charge_origine": None}
                    for step in missing_post_ops
                ],
                "division": of_to_schedule.division or "",
                "horizon": of_to_schedule.horizon_days,
                "category": of_to_schedule.category or ""
            })
            scheduled_ofs.append(of_to_schedule)
            return of_to_schedule

        # ============================================================
        # Mode fini : on planifie les opérations avec capacité réelle
        # ============================================================
        advance_d = get_advance_days(of_to_schedule)
        retard_d = get_retard_days(of_to_schedule)
        earliest_allowed = need_dt - timedelta(days=advance_d)
        latest_allowed = need_dt + timedelta(days=retard_d)

        prod_norm = of_to_schedule.product_id
        qty = of_to_schedule.quantity

        # --- Vérification du stock ---
        stock_ok, components_ready_dt, allocations, stock_reason = check_component_availability(
            prod_norm, qty, ignore_of_id=of_to_schedule.id
        )
        if not stock_ok:
            status = "ECHOUE(stock insuffisant)"
            of_to_schedule.status = status
            of_to_schedule.scheduled_start_date = None
            of_to_schedule.scheduled_end_date = None
            smoothing_items.append(_make_failure_item(of_to_schedule, group_id_str, status, stock_reason))
            scheduled_ofs.append(of_to_schedule)
            if enable_logging:
                log_failure(of_to_schedule.id, "STOCK", "", stock_reason, None, None, None, of_to_schedule.division, of_to_schedule.num_plan)
            return of_to_schedule

        # --- Vérification que tous les postes existent (sauf si ignore_missing_posts) ---
        if not ignore_missing_posts:
            missing_posts = set()
            for step in ops:
                for alt in step.alternatives:
                    if alt.post_id not in posts_map:
                        missing_posts.add(alt.post_id)
            if missing_posts:
                status = "ECHOUE(poste inexistant)"
                of_to_schedule.status = status
                of_to_schedule.scheduled_start_date = None
                of_to_schedule.scheduled_end_date = None
                smoothing_items.append(_make_failure_item(
                    of_to_schedule, group_id_str, status,
                    f"Postes manquants: {', '.join(missing_posts)}"
                ))
                scheduled_ofs.append(of_to_schedule)
                if enable_logging:
                    for p in missing_posts:
                        log_failure(
                            of_to_schedule.id,
                            "VERIFICATION",
                            p,
                            f"Poste inexistant dans la base de capacité ({p})",
                            None, None, None,
                            of_to_schedule.division,
                            of_to_schedule.num_plan
                        )
                return of_to_schedule

        # --- Si aucune opération ---
        if not ops:
            status = "ECHOUE(poste indispo)"
            of_to_schedule.status = status
            of_to_schedule.scheduled_start_date = None
            of_to_schedule.scheduled_end_date = None
            smoothing_items.append(_make_failure_item(of_to_schedule, group_id_str, status, "No operations"))
            scheduled_ofs.append(of_to_schedule)
            if enable_logging:
                log_failure(of_to_schedule.id, "AUCUNE_OPERATION", "", "Aucune opération trouvée", None, None, None, of_to_schedule.division, of_to_schedule.num_plan)
            return of_to_schedule

        clear_of_schedules(of_to_schedule.id, ops)

        # --- Génération des semaines candidates ---
        week_starts = candidate_week_starts(need_dt, advance_d, retard_d, week_sort_order)
        chosen_sched = None
        failure_reason = None

        _do_dbg = enable_logging

        for ws in week_starts:
            if ws > latest_allowed or ws < earliest_allowed - timedelta(days=7):
                continue
            start_search = max(ws, earliest_allowed)
            if components_ready_dt is not None:
                start_search = max(start_search, components_ready_dt)
            # En mode fini, on garde la limite de 14 jours par semaine
            week_end_bound = min(ws + timedelta(days=14), latest_allowed)

            chain_last_end = None
            temp_sched = []
            ok = True
            for step in ops:
                context = {
                    'start_search': start_search,
                    'chain_last_end': chain_last_end,
                    'max_gap_seconds': max_gap_seconds,
                    'latest_allowed': week_end_bound,
                    'of_id': of_to_schedule.id,
                    'division': of_to_schedule.division,
                    'need_date': need_dt,
                    'week_start': ws,
                    'num_plan': of_to_schedule.num_plan,
                    'group_context': False,
                    '_dbg': _do_dbg,
                }
                selected = select_best_alternative(step, context)
                if selected[0] is None:
                    ok = False
                    failure_reason = selected[4]  # "poste_absent", "capacite_insuffisante", etc.
                    break
                alt, post, s_dt, e_dt, _ = selected
                temp_sched.append((step, post, s_dt, e_dt))
                chain_last_end = e_dt
                start_search = e_dt
            if ok and temp_sched:
                chosen_sched = temp_sched
                break

        if chosen_sched:
            for step, post, s_dt, e_dt in chosen_sched:
                post.book_slot(s_dt, e_dt, of_to_schedule.id + "_" + step.operation_name)
            start_dt = chosen_sched[0][2]
            end_dt = chosen_sched[-1][3]
            of_to_schedule.scheduled_start_date = start_dt
            of_to_schedule.scheduled_end_date = end_dt
            if start_dt.date() <= need_dt.date():
                status = "OUI"
            elif need_dt.date() < start_dt.date() <= latest_allowed.date():
                status = "NON"
            else:
                status = "ECHOUE"
            of_to_schedule.status = status
            commit_allocations(allocations, of_to_schedule.id, start_dt)
            if qty > 0:
                add_produced_lot(prod_norm, end_dt, qty, of_to_schedule.id)

            retard_jours = days_delay_if_late(end_dt, need_dt) if status == "NON" else 0
            avance_jours = (need_dt - start_dt).days if start_dt < need_dt else 0

            group_id_final = get_display_group_id(of_to_schedule, group_id_str)

            smoothing_items.append({
                "of_id": of_to_schedule.id, "product_id": of_to_schedule.raw_product_id,
                "designation": of_to_schedule.designation, "group_id": group_id_final,
                "qty_besoin": qty_besoin_raw(of_to_schedule),
                "date_x3": of_to_schedule.original_need_date.strftime("%Y-%m-%d"),
                "need_date": need_dt.strftime("%Y-%m-%d"),
                "scheduled_start": dt_to_str(start_dt), "scheduled_end": dt_to_str(end_dt),
                "status": status, "retard_jours": retard_jours, "avance_jours": avance_jours,
                "num_plan": of_to_schedule.num_plan, "priority": of_to_schedule.priority,
                "operations": [
                    {"operation": step.operation_name, "post_id": post.id, "start": dt_to_str(s_dt), "end": dt_to_str(e_dt),
                     "charge": op_charge_for_post(step, post.id),
                     "charge_origine": op_charge_origine_for_post(step, post.id)}
                    for step, post, s_dt, e_dt in chosen_sched
                ],
                "division": of_to_schedule.division or "",
                "horizon": of_to_schedule.horizon_days,
                "category": of_to_schedule.category or ""
            })
        else:
            # Déterminer le statut en fonction de la raison d'échec
            if failure_reason == "poste_absent":
                status = "ECHOUE(poste inexistant)"
            elif failure_reason == "capacite_insuffisante":
                status = "ECHOUE(semaine pleine)"
            elif failure_reason == "ecart_trop_grand":
                status = "ECHOUE(ecart trop grand)"
            elif failure_reason == "depassement_fenetre":
                status = "ECHOUE(fenetre depassee)"
            else:
                status = "ECHOUE(poste indispo)"
            of_to_schedule.status = status
            of_to_schedule.scheduled_start_date = None
            of_to_schedule.scheduled_end_date = None
            smoothing_items.append(_make_failure_item(of_to_schedule, group_id_str, status, f"Raison: {failure_reason}"))
            if enable_logging:
                log_failure(of_to_schedule.id, "GLOBAL", "", f"Aucune semaine candidate: {failure_reason}", None, None, None, of_to_schedule.division, of_to_schedule.num_plan)

        scheduled_ofs.append(of_to_schedule)
        return of_to_schedule

    # File d'attente triee par priorite
    planning_queue = sorted(
        all_ofs_with_groups,
        key=lambda of: (
            of.priority is None,
            of.priority if of.priority is not None else float('inf'),
            of.need_date,
            -product_level.get(of.product_id, 0),
        )
    )

    for of_to_schedule in planning_queue:
        gid = of_to_schedule.assigned_group_id
        _schedule_of_core(
            of_to_schedule, group_id_str=gid if gid else "INDIVIDUEL", group_context=False,
            gp_last_end=None, tmp_post_end=None
        )

    final_by_id = {of.id: of for of in scheduled_ofs}
    updated_all = [final_by_id.get(orig.id, orig) for orig in all_ofs_with_groups]

    # ---- Écriture des logs en tables Delta ----
    if LOG_LEVEL == "detailed" and LOG_FILTER != "none" and failure_records:
        schema = StructType([
            StructField("OF_Numero", StringType(), True),
            StructField("Operation", StringType(), True),
            StructField("Poste", StringType(), True),
            StructField("Typologie_Echec", StringType(), True),
            StructField("Categorie_Raison", StringType(), True),
            StructField("Date_debut_tentative", TimestampType(), True),
            StructField("Date_fin_tentative", TimestampType(), True),
            StructField("Semaine_candidate", DateType(), True),
            StructField("Statut_final", StringType(), True),
            StructField("Division", StringType(), True),
            StructField("Composant_ou_poste_concerne", StringType(), True),
            StructField("Nb_tentatives", IntegerType(), True),
            StructField("Num_plan", IntegerType(), True)
        ])
        if params.get("id_sim") is not None:
            schema.add(StructField("ID_SIM", StringType(), True))
        df_fail = spark.createDataFrame(failure_records, schema=schema)
        df_fail = df_fail.withColumn("Cle_Poste_Semaine",
                                     F.concat(F.col("Poste"), F.lit("_"), F.col("Semaine_candidate")))
        df_fail = df_fail.withColumn("Operation", F.substring("Operation", 1, 255)) \
                         .withColumn("Composant_ou_poste_concerne", F.substring("Composant_ou_poste_concerne", 1, 255))
        df_fail = df_fail.repartition(20)
        df_fail.write.mode(LOG_WRITE_MODE).option("mergeSchema", "true").format("delta").saveAsTable(OUT_FAILURE_DETAILS)
        print(f"[LOG] Table {OUT_FAILURE_DETAILS} écrite avec {df_fail.count()} enregistrements")

        df_typologie = df_fail.groupBy("Typologie_Echec").count().withColumnRenamed("count", "Nb_occurrences") \
                              .orderBy(F.desc("Nb_occurrences"))
        df_typologie.write.mode(LOG_WRITE_MODE).option("mergeSchema", "true").format("delta").saveAsTable(OUT_FAILURE_SUMMARY_TYPOLOGIE)
        print(f"[LOG] Table {OUT_FAILURE_SUMMARY_TYPOLOGIE} écrite")

        df_top = df_fail.groupBy("Composant_ou_poste_concerne").count().withColumnRenamed("count", "Nb_occurrences") \
                        .orderBy(F.desc("Nb_occurrences"))
        df_top.write.mode(LOG_WRITE_MODE).option("mergeSchema", "true").format("delta").saveAsTable(OUT_FAILURE_SUMMARY_TOP)
        print(f"[LOG] Table {OUT_FAILURE_SUMMARY_TOP} écrite")
    else:
        print("[LOG] Logs désactivés (LOG_FILTER='none' ou LOG_LEVEL != 'detailed')")

    capture = params.get("capture")
    if capture is not None:
        capture["smoothing_items"] = smoothing_items
        capture["stock_movements"] = stock_movements
        capture["posts_map"] = posts_map
        capture["groups"] = groups

    return updated_all


# ---------------------------------------------------------------------
# DETECTION AUTOMATIQUE DU PREFIXE DE TABLE
# ---------------------------------------------------------------------
def _detect_table_prefix(spark, probe_table="sim_of_work"):
    candidates = [
        f"{LAKEHOUSE_NAME}.",
        f"{LAKEHOUSE_NAME}.dbo.",
        "",
    ]
    last_err = None
    for prefix in candidates:
        full_name = f"{prefix}{probe_table}"
        try:
            spark.table(full_name).take(1)
            print(f"[Lakehouse] Prefixe de table detecte : '{prefix}' (ex: {full_name})")
            return prefix
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(
        f"Impossible de trouver la table '{probe_table}' sous aucune des formes testees: "
        f"{[c + probe_table for c in candidates]}.\n"
        f"-> Verifie que le lakehouse '{LAKEHOUSE_NAME}' est bien attache a ce notebook "
        f"(panneau Explorateur a gauche -> Lakehouses -> Ajouter -> {LAKEHOUSE_NAME}), "
        f"et que le nom de la table/du schema est correct.\n"
        f"Derniere erreur: {last_err}"
    )


# ---------------------------------------------------------------------
# CHARGEMENT DEPUIS LES TABLES LAKEHOUSE (LECTURE SEULE)
# ---------------------------------------------------------------------
def load_bom_from_spark(spark):
    df = spark.table(TBL_BOM).select(
        "ParentProductID", "ChildProductID", "QuantityChildPerParent",
        "ChildBOMLevel", "ParentBOMLevel"
    )
    rows = df.collect()
    bom_entries = []
    for r in rows:
        if not r["ParentProductID"] or not r["ChildProductID"]:
            continue
        bom_entries.append(BOMEntry(
            parent_product_id=r["ParentProductID"],
            child_product_id=r["ChildProductID"],
            quantity_child_per_parent=r["QuantityChildPerParent"] if r["QuantityChildPerParent"] is not None else 0.0,
            child_bom_level=r["ChildBOMLevel"] if r["ChildBOMLevel"] is not None else 0,
            parent_bom_level=r["ParentBOMLevel"],
        ))
    print(f"[Load] {len(bom_entries)} lignes BOM chargees depuis {TBL_BOM}")
    return bom_entries


def load_article_params_from_spark(spark):
    df = spark.table(TBL_X3ITM).select("ARTICLE_0", "CST1_0", "CST2_0", "CST3_0", "CATEGORIE_0", "FAMSTA3_0")
    rows = df.collect()
    params_map = {}
    for r in rows:
        article = r["ARTICLE_0"]
        if not article:
            continue
        norm_art = _norm(article)
        cst1 = float(r["CST1_0"]) if r["CST1_0"] is not None else None
        cst2 = float(r["CST2_0"]) if r["CST2_0"] is not None else None
        cst3 = float(r["CST3_0"]) if r["CST3_0"] is not None else None
        categorie = str(r["CATEGORIE_0"]).strip() if r["CATEGORIE_0"] is not None else ""
        division = str(r["FAMSTA3_0"]).strip() if r["FAMSTA3_0"] is not None else ""
        params_map[norm_art] = (cst1, cst2, cst3, categorie, division)
    print(f"[Load] {len(params_map)} articles parametres charges depuis {TBL_X3ITM} (incluant CATEGORIE_0 et FAMSTA3_0)")
    return params_map


def load_ofs_and_operations_from_spark(spark, article_params_map):
    df = spark.table(TBL_OF_WORK).select(
        "OF_Numero", "Article", "QteOF", "DateFinOF", "DesignationArticle",
        "Operation", "DesignationOperation", "Poste", "ChargeCalculee", "Div"
    )
    rows = df.collect()

    header_by_of = {}
    ops_rows_by_of = defaultdict(list)

    for r in rows:
        of_id = r["OF_Numero"]
        if not of_id:
            continue
        ops_rows_by_of[of_id].append(r)
        if of_id not in header_by_of:
            header_by_of[of_id] = r
        elif not header_by_of[of_id]["Div"] and r["Div"]:
            header_by_of[of_id] = r

    # Priorites
    priority_map = {}
    try:
        df_prior = spark.table(TBL_X3OF).select("VCRNUM_0", "PRIORITE_0")
        rows_prior = df_prior.collect()
        for r in rows_prior:
            of_num = r["VCRNUM_0"]
            if of_num:
                priority_map[_norm(of_num)] = r["PRIORITE_0"]
        print(f"[Load] {len(priority_map)} priorites chargees depuis {TBL_X3OF}")
        if priority_map:
            sample = list(priority_map.items())[:5]
            print(f"[Load] Exemple de priorites: {sample}")
        else:
            print("[Load] Aucune priorite chargee (la table est vide ou les cles ne correspondent pas)")
    except Exception as e:
        print(f"[Load] Impossible de charger {TBL_X3OF}: {e}")

    all_ofs = []
    skipped_no_date = 0
    for of_id, header in header_by_of.items():
        article = header["Article"] or ""
        if article.startswith("PF"):
            product_type = "PF"
        elif article.startswith("SF"):
            product_type = "SF"
        elif article.startswith("PS"):
            product_type = "PS"
        else:
            product_type = "UNKNOWN"

        date_fin = header["DateFinOF"]
        if date_fin is None:
            skipped_no_date += 1
            continue

        norm_article = _norm(article)
        adv, ret, hor, categorie, division_itm = article_params_map.get(norm_article, (None, None, None, "", ""))
        if hor is None:
            hor = HORIZON_H_DAYS

        of = ManufacturingOrder(
            id=of_id,
            designation=header["DesignationArticle"] or "",
            product_id=article,
            product_type=product_type,
            bom_level=0,
            need_date_str=date_fin,
            quantity=header["QteOF"] if header["QteOF"] is not None else 0.0,
            fg="", cat="", us="", fs="",
            horizon_days=hor,
            retard_days=ret,
            advance_days=adv,
            priority=priority_map.get(_norm(of_id)),
            category=categorie
        )
        # La date originale est déjà stockée dans original_need_date via le constructeur
        # Division = FAMSTA3_0 de X3ITM (par article) ; repli sur Div de sim_of_work si absente
        of.division = division_itm or header["Div"] or ""
        all_ofs.append(of)

    if skipped_no_date:
        print(f"[Load] {skipped_no_date} OFs ignores (DateFinOF manquante)")

    operations_map = defaultdict(list)
    skipped_ops_no_poste = 0
    for of_id, op_rows in ops_rows_by_of.items():
        key = _norm(of_id)
        op_rows_sorted = sorted(op_rows, key=lambda r: (int(r["Operation"]) if r["Operation"] is not None else 0))
        for r in op_rows_sorted:
            poste_raw = r["Poste"]
            if not poste_raw:
                skipped_ops_no_poste += 1
                continue
            seq = int(r["Operation"]) if r["Operation"] is not None else 0
            hours = float(r["ChargeCalculee"]) if r["ChargeCalculee"] is not None else 0.0
            step = OperationStep(
                sequence=seq,
                operation_name=r["DesignationOperation"] or f"OP{seq}",
                alternatives=[OperationAlternative(post_id=norm_code(poste_raw), standard_time_hours=hours, priority=1)],
            )
            operations_map[key].append(step)

    for key in operations_map:
        operations_map[key].sort(key=lambda step: step.sequence)

    if skipped_ops_no_poste:
        print(f"[Load] {skipped_ops_no_poste} lignes d'operation ignorees (Poste manquant)")

    print(f"[Load] {len(all_ofs)} OFs charges depuis {TBL_OF_WORK} "
          f"({len(operations_map)} OFs avec au moins 1 operation exploitable)")

    nb_priorized = sum(1 for of in all_ofs if of.priority is not None)
    print(f"[DEBUG] OFs avec priorité non nulle : {nb_priorized} / {len(all_ofs)}")

    cats = set(of.category for of in all_ofs if of.category)
    print(f"[DEBUG] Catégories présentes : {cats}")
    nb_118 = sum(1 for of in all_ofs if of.category == "118")
    print(f"[DEBUG] OFs avec catégorie '118' : {nb_118}")

    return all_ofs, operations_map


# =====================================================================
# MODIFICATION : Suppression du vote → détection OR (si une ligne =1 => goulot)
# =====================================================================
def load_posts_from_spark(spark):
    df = spark.table(TBL_POSTE_CAPA).select("Poste", "Semaine", "capacite", "Goulot", "Designation")
    rows = df.collect()

    def parse_goulot(v):
        s = str(v or "").strip().lower()
        return s in ("1", "oui", "true", "yes", "y", "o")

    raw_capacities = defaultdict(list)
    post_is_bottleneck = defaultdict(bool)   # False par défaut (non‑goulot)
    post_names = {}
    all_mondays = []

    for r in rows:
        raw_pid = r["Poste"]
        if not raw_pid:
            continue
        pid = norm_code(raw_pid)
        sem = r["Semaine"]
        cap = r["capacite"]
        if sem is None or cap is None:
            continue
        monday = sem.date() if hasattr(sem, "date") else sem
        raw_capacities[pid].append((monday, float(cap)))
        all_mondays.append(monday)

        # PLUS DE VOTE : si une seule ligne a Goulot=1, le poste est goulot
        if parse_goulot(r["Goulot"]):
            post_is_bottleneck[pid] = True

        if pid not in post_names and r["Designation"]:
            post_names[pid] = r["Designation"]

    if not raw_capacities:
        print(f"[Load] ATTENTION: aucune capacite trouvee dans {TBL_POSTE_CAPA}")
        return {}

    base_monday = min(all_mondays)
    posts_map = {}
    for pid, capacities in raw_capacities.items():
        weekly_cap = {}
        for monday, cap in capacities:
            week_idx = 1 + (monday - base_monday).days // 7
            weekly_cap[week_idx] = cap

        # Détermination directe (pas de vote)
        is_bottleneck = post_is_bottleneck.get(pid, False)

        post = Post(
            id=pid,
            name=post_names.get(pid, pid),
            weekly_capacity_by_week=weekly_cap,
            is_bottleneck=is_bottleneck,
            override_weeks=None,
        )
        post.set_week0_monday(base_monday)
        for _arret_deb, _arret_fin in SIM_ARRETS:
            post.add_unavailable_period(_arret_deb, _arret_fin)
        posts_map[pid] = post

    print(f"[Load] {len(posts_map)} postes charges depuis {TBL_POSTE_CAPA} (base Monday = {base_monday})")
    if SIM_ARRETS:
        print(f"[Load] Arrets appliques a tous les postes : {SIM_ARRETS}")
    return posts_map


# ---------------------------------------------------------------------
# ORCHESTRATION D'UN RUN (chargement + groupage + planification)
# ---------------------------------------------------------------------

def load_inputs():
    """(Re)charge les OFs/operations/BOM/postes depuis Spark, avec le filtre d'horizon."""
    article_params = load_article_params_from_spark(spark)
    ofs, ops_map = load_ofs_and_operations_from_spark(spark, article_params)

    if HORIZON_DEB is not None and HORIZON_FIN is not None:
        try:
            deb_date = try_parse_date(HORIZON_DEB)
            fin_date = try_parse_date(HORIZON_FIN)
            original_count = len(ofs)
            ofs = [of for of in ofs if deb_date <= of.need_date <= fin_date]
            print(f"[Filtre] OFs retenus : {len(ofs)} sur {original_count} (plage {deb_date.date()} - {fin_date.date()})")
        except Exception as e:
            print(f"[Filtre] Erreur lors du parsing des dates : {e}. Aucun filtre appliqué.")
    else:
        print("[Filtre] Aucun filtre de plage de dates appliqué (tous les OFs sont traités).")

    bom = load_bom_from_spark(spark)

    # Evolution de la quantite a fabriquer par article (scenario de demande)
    apply_quantity_evolution(ofs, ops_map, bom, globals().get("EVOL_QTE_MAP") or {})

    posts = load_posts_from_spark(spark)
    return ofs, ops_map, bom, posts


def run_planning(force_infinite_capacity, simulation_mode=False, id_sim=None, log_write_mode="overwrite"):
    """Execute un run complet (groupage + lissage/planif) et retourne tout ce qu'il faut
    pour construire les dataframes de sortie."""
    all_ofs, operations_map, bom_data, posts_map = load_inputs()

    if force_infinite_capacity:
        for _p in posts_map.values():
            _p.is_bottleneck = False
        print("[Run] Capacite infinie -> tous les postes forces non-goulot")
    else:
        print("[Run] Capacite finie -> capacite des postes inchangee (SIM_POSTE_CAPA)")

    print(f"[DEBUG] Nb OFs: {len(all_ofs)}")
    print(f"[DEBUG] Nb postes: {len(posts_map)}")
    print(f"[DEBUG] Nb OFs avec operations: {len(operations_map)}")

    # ---- Résolution des vérifications (None = défaut du mode) ----
    def _verif(v, default):
        return default if v is None else bool(v)

    is_inf = bool(simulation_mode)
    verif_stock   = _verif(VERIF_STOCK,       not is_inf)
    verif_ecart   = _verif(VERIF_ECART_OPS,   not is_inf)
    verif_fenetre = _verif(VERIF_FENETRE,     not is_inf) if is_inf else True
    if (not is_inf) and VERIF_FENETRE is False:
        print("[Run][WARN] VERIF_FENETRE=False ignoree en capacite Finie (fenetre toujours appliquee)")
    print(f"[Run] Verifications -> stock={verif_stock} | ecart_ops={verif_ecart} | "
          f"fenetre={verif_fenetre}")

    capture = {}
    # Paramètres par défaut
    params = {
        "advance_retreat_days": ADVANCE_RETREAT_DAYS,
        "max_gap_between_ops_hours": MAX_GAP_BETWEEN_OPS_HOURS,
        "week_sort_order": WEEK_SORT_ORDER,
        "min_week_monday": SIM_LAUNCH_MONDAY,   # plancher "pas de passe" (lundi de lancement)
        "debug_log_path": "/tmp/planning_debug.log",
        "capture": capture,
        "ignore_gap": not verif_ecart,              # VERIF_ECART_OPS
        "ignore_stock": not verif_stock,            # VERIF_STOCK
        "ignore_missing_posts": simulation_mode,    # comportement d'origine (ECHOUE(poste inexistant) en Finie)
        # ignore_window = True selectionne le chemin "Infinie simple" (OF place a sa date de besoin,
        # sans fenetre avance/retard). False = chemin complet (fenetre CST1_0/CST2_0 appliquee).
        "ignore_window": is_inf and not verif_fenetre,
        "id_sim": id_sim,
        "log_write_mode": log_write_mode,
    }

    # Si simulation : pas de plafond en jours (ignore_window s'en charge avec
    # datetime.max, cf. _schedule_of_core), et pas de limite d'écart entre
    # opérations.
    if not verif_ecart:
        params["max_gap_between_ops_hours"] = 10**9  # pas de limite d'écart

    groups, all_ofs_with_groups = run_grouping_algorithm(all_ofs, bom_data, HORIZON_H_DAYS)
    print(f"[DEBUG] Nb groupes formes: {len(groups)}")

    all_ofs_scheduled = smooth_and_schedule_groups(
        groups, all_ofs_with_groups, bom_data, posts_map, operations_map, params
    )
    print(f"[DEBUG] Nb OFs planifies: {len(all_ofs_scheduled)}")

    return all_ofs_scheduled, capture, groups, posts_map


# ---------------------------------------------------------------------
# ECRITURE DES TABLES DE SORTIE S_*
# ---------------------------------------------------------------------

def write_simulation_outputs(all_ofs_scheduled, capture, groups, posts_map, write_mode, id_sim=None):
    """Construit et ecrit exclusivement les tables de sortie S_* de la simulation.
    Les résultats sont ajoutés en append et portent l'ID_SIM du run."""

    # Tables de sortie : S_* uniquement.
    out_grouped_needs = f"{_prefix}S_grouped_needs"
    out_smoothing = f"{_prefix}S_smoothing"
    out_smoothing_ops = f"{_prefix}S_smoothing_operations"
    out_weekly_capacity = f"{_prefix}S_weekly_capacity"
    out_groups_anchors = f"{_prefix}S_groups_anchors"
    out_summary_division_status = f"{_prefix}S_summary_division_status"

    # --- grouped_needs : 1 ligne par OF, groupe + planif -----------------
    grouped_needs_rows = []
    for of_obj in all_ofs_scheduled:
        group_id = of_obj.assigned_group_id if of_obj.assigned_group_id is not None else "INDIVIDUEL"
        if getattr(of_obj, "category", "") == "118":
            group_id = "INDIVIDUEL"
        row = {
            "OF_Numero": of_obj.id,
            "Article": of_obj.raw_product_id,
            "Designation": of_obj.designation,
            "ProductType": of_obj.product_type,
            "Div": of_obj.division or "",
            "Qty": float(of_obj.quantity),
            "DATE_X3": of_obj.original_need_date.strftime("%Y-%m-%d") if of_obj.original_need_date else None,
            "NeedDate": of_obj.need_date.strftime("%Y-%m-%d") if of_obj.need_date else None,
            "GroupID": group_id,
            "ScheduledStart": of_obj.scheduled_start_date.strftime("%Y-%m-%d %H:%M") if of_obj.scheduled_start_date else None,
            "ScheduledEnd": of_obj.scheduled_end_date.strftime("%Y-%m-%d %H:%M") if of_obj.scheduled_end_date else None,
            "Status": of_obj.status,
            "NumPlan": of_obj.num_plan,
            "Priority": of_obj.priority,
            "IndividualProductStock": float(of_obj.individual_product_stock) if of_obj.individual_product_stock is not None else 0.0,
            "Qty_Origine": float(getattr(of_obj, "qty_origine", of_obj.quantity) or 0.0),
            "Coef_Evol": float(getattr(of_obj, "coef_evol", 1.0) or 1.0),
        }
        if id_sim is not None:
            row["ID_SIM"] = id_sim
        grouped_needs_rows.append(row)

    schema_grouped_needs = StructType([
        StructField("OF_Numero", StringType(), True),
        StructField("Article", StringType(), True),
        StructField("Designation", StringType(), True),
        StructField("ProductType", StringType(), True),
        StructField("Div", StringType(), True),
        StructField("Qty", DoubleType(), True),
        StructField("DATE_X3", StringType(), True),
        StructField("NeedDate", StringType(), True),
        StructField("GroupID", StringType(), True),
        StructField("ScheduledStart", StringType(), True),
        StructField("ScheduledEnd", StringType(), True),
        StructField("Status", StringType(), True),
        StructField("NumPlan", IntegerType(), True),
        StructField("Priority", IntegerType(), True),
        StructField("IndividualProductStock", DoubleType(), True),
        StructField("Qty_Origine", DoubleType(), True),
        StructField("Coef_Evol", DoubleType(), True),
    ])
    # Ajout de ID_SIM au schema si necessaire
    if id_sim is not None:
        schema_grouped_needs.add(StructField("ID_SIM", StringType(), True))

    df_grouped_needs = spark.createDataFrame(grouped_needs_rows, schema=schema_grouped_needs)
    df_grouped_needs.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_grouped_needs)
    print(f"[Write] {out_grouped_needs} : {df_grouped_needs.count()} lignes")

    # --- smoothing / smoothing_operations -----------------------------
    smoothing_items = capture.get("smoothing_items", [])
    smoothing_rows = []
    smoothing_ops_rows = []

    def _to_week_start_str(dt_str):
        """'YYYY-MM-DD HH:MM' -> date du lundi de la meme semaine, format 'YYYY-MM-DD' (sans heure)."""
        if not dt_str:
            return dt_str
        try:
            d = datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
        except ValueError:
            d = datetime.strptime(dt_str[:10], "%Y-%m-%d")
        monday = d.date() - timedelta(days=d.weekday())
        return monday.strftime("%Y-%m-%d")
    _of_by_id = {o.id: o for o in all_ofs_scheduled}
    for item in smoothing_items:
        _of_obj = _of_by_id.get(item.get("of_id"))
        srow = {
            "num_plan": item.get("num_plan"),
            "priority": item.get("priority"),
            "group_id": item.get("group_id"),
            "of_id": item.get("of_id"),
            "product_id": item.get("product_id"),
            "designation": item.get("designation"),
            "qty_besoin": item.get("qty_besoin"),
            "date_x3": item.get("date_x3"),
            "need_date": item.get("need_date"),
            # Dates de planification ramenees au lundi de la semaine (S_smoothing uniquement)
            "scheduled_start": _to_week_start_str(item.get("scheduled_start")),
            "scheduled_end": _to_week_start_str(item.get("scheduled_end")),
            "status": item.get("status"),
            "retard_jours": item.get("retard_jours", 0),
            "avance_jours": item.get("avance_jours", 0),
            "division": item.get("division", ""),
            "horizon": item.get("horizon"),
            "category": item.get("category", ""),
            "qty_origine": float(_of_obj.qty_origine) if _of_obj is not None else None,
            "coef_evol": float(_of_obj.coef_evol) if _of_obj is not None else None,
        }
        if id_sim is not None:
            srow["ID_SIM"] = id_sim
        smoothing_rows.append(srow)
        for op in item.get("operations", []):
            op_row = {
                "of_id": item.get("of_id"),
                "num_plan": item.get("num_plan"),
                "operation": op.get("operation"),
                "post_id": op.get("post_id"),
                "start": op.get("start"),
                "end": op.get("end"),
                "charge_calc": op.get("charge"),
                "Charge_Origine": op.get("charge_origine"),
            }
            if id_sim is not None:
                op_row["ID_SIM"] = id_sim
            smoothing_ops_rows.append(op_row)

    schema_smoothing = StructType([
        StructField("num_plan", IntegerType(), True),
        StructField("priority", IntegerType(), True),
        StructField("group_id", StringType(), True),
        StructField("of_id", StringType(), True),
        StructField("product_id", StringType(), True),
        StructField("designation", StringType(), True),
        StructField("qty_besoin", StringType(), True),
        StructField("date_x3", StringType(), True),
        StructField("need_date", StringType(), True),
        StructField("scheduled_start", StringType(), True),
        StructField("scheduled_end", StringType(), True),
        StructField("status", StringType(), True),
        StructField("retard_jours", IntegerType(), True),
        StructField("avance_jours", IntegerType(), True),
        StructField("division", StringType(), True),
        StructField("horizon", DoubleType(), True),
        StructField("category", StringType(), True),
        StructField("qty_origine", DoubleType(), True),
        StructField("coef_evol", DoubleType(), True),
    ])
    if id_sim is not None:
        schema_smoothing.add(StructField("ID_SIM", StringType(), True))

    schema_smoothing_ops = StructType([
        StructField("of_id", StringType(), True),
        StructField("num_plan", IntegerType(), True),
        StructField("operation", StringType(), True),
        StructField("post_id", StringType(), True),
        StructField("start", StringType(), True),
        StructField("end", StringType(), True),
        StructField("charge_calc", DoubleType(), True),
        StructField("Charge_Origine", DoubleType(), True),
    ])
    if id_sim is not None:
        schema_smoothing_ops.add(StructField("ID_SIM", StringType(), True))

    df_smoothing = spark.createDataFrame(smoothing_rows, schema=schema_smoothing) if smoothing_rows \
        else spark.createDataFrame([], schema=schema_smoothing)
    df_smoothing.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_smoothing)
    print(f"[Write] {out_smoothing} : {df_smoothing.count()} lignes")

    df_smoothing_ops = spark.createDataFrame(smoothing_ops_rows, schema=schema_smoothing_ops) if smoothing_ops_rows \
        else spark.createDataFrame([], schema=schema_smoothing_ops)
    # Charge = ChargeCalculee (h, source sim_of_work) de CHAQUE operation (une ligne = une operation).
    # Pas d'agregation par poste : SUM(Charge) par OF / poste / semaine redonne exactement sim_of_work.
    df_smoothing_ops = df_smoothing_ops.withColumnRenamed("charge_calc", "Charge")
    df_smoothing_ops.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_smoothing_ops)
    print(f"[Write] {out_smoothing_ops} : {df_smoothing_ops.count()} lignes")

    # --- weekly_capacity : capacite vs charge par poste/semaine ----------
    weekly_capacity_rows = []
    posts_map_final = capture.get("posts_map", posts_map)
    for post in posts_map_final.values():
        base = getattr(post, "planning_start_monday", None)
        if base is None:
            continue
        is_bottleneck = bool(getattr(post, "is_bottleneck", True))
        first = post._first_week_index if post._first_week_index is not None else 1
        last = post._last_week_index if post._last_week_index is not None else (
            max(post.weekly_load_hours.keys()) if post.weekly_load_hours else 1
        )
        for wk_index in range(first, last + 1):
            planned = post.weekly_load_hours.get(wk_index, 0.0)
            week_start = base + timedelta(days=7 * (wk_index - 1))
            week_end = week_start + timedelta(days=6)

            if not is_bottleneck:
                # Affichage uniquement : on montre l'horaire initial (SIM_POSTE_CAPA),
                # meme en capacite infinie ; la planification, elle, reste non contrainte.
                allowed_display = float(post.weekly_capacity_by_week.get(wk_index, 0.0))
                row = {
                    "PostID": post.id, "PostName": post.name, "Goulot": 1 if is_bottleneck else 0,
                    "WeekIndex": wk_index,
                    "WeekStartDate": week_start.strftime("%Y-%m-%d"),
                    "WeekEndDate": week_end.strftime("%Y-%m-%d"),
                    "AllowedHours": float(allowed_display), "PlannedHours": float(planned),
                    "DeltaHours": None, "RespectCapacity": "N/A",
                }
                if id_sim is not None:
                    row["ID_SIM"] = id_sim
                weekly_capacity_rows.append(row)
                continue

            dt_for_wk = datetime.combine(week_start, time.min)
            allowed = float(post._allowed_hours_for_week(dt_for_wk))
            delta = allowed - float(planned)
            respect = "OUI" if planned <= allowed + 1e-6 else "NON"
            row = {
                "PostID": post.id, "PostName": post.name, "Goulot": 1 if is_bottleneck else 0,
                "WeekIndex": wk_index,
                "WeekStartDate": week_start.strftime("%Y-%m-%d"),
                "WeekEndDate": week_end.strftime("%Y-%m-%d"),
                "AllowedHours": float(allowed), "PlannedHours": float(planned),
                "DeltaHours": float(delta), "RespectCapacity": respect,
            }
            if id_sim is not None:
                row["ID_SIM"] = id_sim
            weekly_capacity_rows.append(row)

    schema_weekly_capacity = StructType([
        StructField("PostID", StringType(), True),
        StructField("PostName", StringType(), True),
        StructField("Goulot", IntegerType(), True),
        StructField("WeekIndex", IntegerType(), True),
        StructField("WeekStartDate", StringType(), True),
        StructField("WeekEndDate", StringType(), True),
        StructField("AllowedHours", DoubleType(), True),
        StructField("PlannedHours", DoubleType(), True),
        StructField("DeltaHours", DoubleType(), True),
        StructField("RespectCapacity", StringType(), True),
    ])
    if id_sim is not None:
        schema_weekly_capacity.add(StructField("ID_SIM", StringType(), True))

    df_weekly_capacity = spark.createDataFrame(weekly_capacity_rows, schema=schema_weekly_capacity) if weekly_capacity_rows \
        else spark.createDataFrame([], schema=schema_weekly_capacity)
    df_weekly_capacity.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_weekly_capacity)
    print(f"[Write] {out_weekly_capacity} : {df_weekly_capacity.count()} lignes")

    # --- groups_anchors : liste des groupes et leur OF ancre (hors categorie 118) ---
    groups_anchors_rows = []
    for group in groups:
        if not group.ofs:
            continue
        anchor = group.ofs[0]
        row = {
            "GroupID": group.id,
            "Anchor_OF_Numero": anchor.id,
            "Anchor_Product": anchor.raw_product_id,
            "Anchor_NeedDate": anchor.need_date.strftime("%Y-%m-%d") if anchor.need_date else None,
            "Anchor_OriginalNeedDate": anchor.original_need_date.strftime("%Y-%m-%d") if anchor.original_need_date else None,
            "WindowStart": group.time_window_start.strftime("%Y-%m-%d") if group.time_window_start else None,
            "WindowEnd": group.time_window_end.strftime("%Y-%m-%d") if group.time_window_end else None,
            "NbOFs": len(group.ofs),
            "Anchor_Category": getattr(anchor, "category", ""),
            "Anchor_Division": getattr(anchor, "division", ""),
        }
        if id_sim is not None:
            row["ID_SIM"] = id_sim
        groups_anchors_rows.append(row)

    schema_groups_anchors = StructType([
        StructField("GroupID", StringType(), True),
        StructField("Anchor_OF_Numero", StringType(), True),
        StructField("Anchor_Product", StringType(), True),
        StructField("Anchor_NeedDate", StringType(), True),
        StructField("Anchor_OriginalNeedDate", StringType(), True),
        StructField("WindowStart", StringType(), True),
        StructField("WindowEnd", StringType(), True),
        StructField("NbOFs", IntegerType(), True),
        StructField("Anchor_Category", StringType(), True),
        StructField("Anchor_Division", StringType(), True),
    ])
    if id_sim is not None:
        schema_groups_anchors.add(StructField("ID_SIM", StringType(), True))

    df_groups_anchors = spark.createDataFrame(groups_anchors_rows, schema=schema_groups_anchors) if groups_anchors_rows \
        else spark.createDataFrame([], schema=schema_groups_anchors)
    df_groups_anchors.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_groups_anchors)
    print(f"[Write] {out_groups_anchors} : {df_groups_anchors.count()} lignes")

    # --- produced_quantities : quantite demandee vs quantite produite,
    # par produit et par mois de fin planifiee (Annee_Mois), pour ce run -----
    out_produced_quantities = f"{_prefix}S_produced_quantities"

    df_status_clean = df_grouped_needs.withColumn(
        "status_clean",
        F.when(F.col("Status").contains("("), F.split(F.col("Status"), "\\(")[0])
         .otherwise(F.col("Status"))
    ).withColumn(
        # Mois de fin planifiee (meme valeur que scheduled_end de S_smoothing)
        "Annee_Mois",
        F.date_format(F.to_timestamp(F.col("ScheduledEnd"), "yyyy-MM-dd HH:mm"), "yyyy-MM")
    )
    df_produced_quantities = df_status_clean.groupBy("Article", "Designation", "ProductType", "Div", "Annee_Mois").agg(
        F.sum(F.when(F.col("status_clean").isin("OUI", "NON"), F.col("Qty")).otherwise(F.lit(0.0))).alias("QuantityProduced"),
        F.sum("Qty").alias("QuantityDemandee"),
        F.sum("Qty_Origine").alias("QuantityDemandee_Origine"),
        F.sum(F.when(F.col("status_clean").isin("OUI", "NON"), F.lit(1)).otherwise(F.lit(0))).cast(IntegerType()).alias("NbOF_Produits"),
        F.sum(F.when(F.col("status_clean") == "NON", F.lit(1)).otherwise(F.lit(0))).cast(IntegerType()).alias("NbOF_ProduitsEnRetard"),
        F.count(F.lit(1)).cast(IntegerType()).alias("NbOF_Total"),
    )
    df_produced_quantities = df_produced_quantities.withColumn(
        "TauxService",
        F.when(F.col("QuantityDemandee") > 0,
               F.col("QuantityProduced") / F.col("QuantityDemandee") * 100)
         .otherwise(F.lit(None))
    )
    if id_sim is not None:
        df_produced_quantities = df_produced_quantities.withColumn("ID_SIM", F.lit(id_sim))
    df_produced_quantities = df_produced_quantities.orderBy("Article", "Annee_Mois")

    df_produced_quantities.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_produced_quantities)
    print(f"[Write] {out_produced_quantities} : {df_produced_quantities.count()} lignes")

    # --- stock_movements : tracabilite du stock SF (entrees = lots produits, sorties = consommation par les PF) ---
    out_stock_movements = f"{_prefix}S_stock_movements"
    _moves = (capture or {}).get("stock_movements") or []
    if _moves:
        _rows = [(m["article"], m["of_id"], m["mouvement"], m["qty"], m["date"], m["of_source"]) for m in _moves]
        _schema = StructType([
            StructField("Article", StringType(), True),
            StructField("OF", StringType(), True),
            StructField("Mouvement", StringType(), True),
            StructField("Qte", DoubleType(), True),
            StructField("Date", TimestampType(), True),
            StructField("OF_Source_Lot", StringType(), True),
        ])
        df_moves = spark.createDataFrame(_rows, schema=_schema)
        # Stock cumule par article, dans l'ordre chronologique (entree avant sortie a date egale)
        _w_stock = Window.partitionBy("Article").orderBy(
            F.col("Date"),
            F.when(F.col("Mouvement") == "entree", 0).otherwise(1),
            F.col("OF")
        ).rowsBetween(Window.unboundedPreceding, Window.currentRow)
        df_moves = df_moves.withColumn("Stock", F.sum("Qte").over(_w_stock))
        if id_sim is not None:
            df_moves = df_moves.withColumn("ID_SIM", F.lit(id_sim))
        df_moves = df_moves.select(
            *(["ID_SIM"] if id_sim is not None else []),
            "Article", "OF", "Mouvement", "Qte", "Stock", "Date", "OF_Source_Lot"
        ).orderBy("Article", "Date")
        df_moves.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_stock_movements)
        print(f"[Write] {out_stock_movements} : {df_moves.count()} lignes")
    else:
        print("[Write] Aucun mouvement de stock a ecrire (verification de stock desactivee ou aucun lot)")

    # --- summary_division_status : Nb OFs uniques par division et statut ---
    df_smoothing_read = spark.table(out_smoothing)
    if id_sim is not None:
        # Filtrer par ID_SIM (unique) pour n'avoir que le run courant
        df_smoothing_read = df_smoothing_read.filter(F.col("ID_SIM") == F.lit(id_sim))

    df_smoothing_clean = df_smoothing_read.withColumn("status_clean",
        F.when(F.col("status").contains("("), F.split(F.col("status"), "\\(")[0])
         .otherwise(F.col("status"))
    )
    df_status = df_smoothing_clean.filter(F.col("division").isNotNull()) \
                                  .groupBy("division", "status_clean").count()
    df_pivot = df_status.groupBy("division").pivot("status_clean", ["OUI", "NON", "ECHOUE"]).sum("count")
    df_pivot = df_pivot.fillna(0, subset=["OUI", "NON", "ECHOUE"])
    df_pivot = df_pivot.withColumn("Total", F.col("OUI") + F.col("NON") + F.col("ECHOUE"))
    df_pivot = df_pivot.orderBy("division")

    # Ajout de ID_SIM si nécessaire (et suppression de SimDesc, SimUser, SimRunAt)
    if id_sim is not None:
        df_pivot = df_pivot.withColumn("ID_SIM", F.lit(id_sim))

    df_pivot.write.mode(write_mode).option("mergeSchema", "true").format("delta").saveAsTable(out_summary_division_status)
    print(f"[OK] Table {out_summary_division_status} créée")
    df_pivot.show(truncate=False)


# ---------------------------------------------------------------------
# INITIALISATION
# ---------------------------------------------------------------------
_prefix = _detect_table_prefix(spark, "sim_of_work")

TBL_OF_WORK = f"{_prefix}sim_of_work"
TBL_BOM = f"{_prefix}sim_nom_child"
TBL_POSTE_CAPA = f"{_prefix}SIM_POSTE_CAPA"
TBL_X3OF = f"{_prefix}X3OF"
TBL_X3ITM = f"{_prefix}X3ITM"
TBL_SIMULATION = f"{_prefix}Simulation"

if CAPACITE_MODE not in ("Finie", "Infinie"):
    raise ValueError("CAPACITE_MODE doit être 'Finie' ou 'Infinie'.")

# Evolution de quantite par article : lue une seule fois (dictionnaire + table optionnelle)
EVOL_QTE_MAP = load_evol_qte_map(spark, f"{_prefix}{EVOL_QTE_TABLE}" if EVOL_QTE_TABLE else None)
EVOL_QTE_DESC = describe_evol_qte(EVOL_QTE_MAP)

# Génération d'un ID_SIM unique pour ce run.
_yymm = datetime.now().strftime("%y%m")
try:
    _existing_max = (
        spark.table(TBL_SIMULATION)
        .filter(F.col("ID_SIM").startswith(f"SIM{_yymm}"))
        .agg(F.max(F.substring(F.col("ID_SIM"), 8, 5)).alias("m"))
        .collect()[0]["m"]
    )
    _next_seq = int(_existing_max) + 1 if _existing_max is not None else 1
except Exception:
    _next_seq = 1

ID_SIM = f"SIM{_yymm}{_next_seq:05d}"

# Les logs S_* sont append et portent le même ID_SIM.
LOG_WRITE_MODE = "append"

print(f"[SIM] Prefixe Lakehouse : '{_prefix}'")
print(f"[SIM] ID_SIM : {ID_SIM}")
print(f"[SIM] CAPACITE_MODE : {CAPACITE_MODE}")
print(f"[SIM] MODE_PLANIF : {MODE_PLANIF}")
print(f"[SIM] LANCEMENT : {SIM_LAUNCH_DATE or 'date du jour'} (lundi plancher = {SIM_LAUNCH_MONDAY.date()})")
print(f"[SIM] DESC : {SIM_DESC}")
print(f"[SIM] USER : {SIM_USER}")
print(f"[SIM] EVOLUTION QTE : {EVOL_QTE_DESC}")

# ---------------------------------------------------------------------
# EN-TÊTE DE LA SIMULATION
# ---------------------------------------------------------------------
_deb = try_parse_date(HORIZON_DEB) if HORIZON_DEB else None
_fin = try_parse_date(HORIZON_FIN) if HORIZON_FIN else None
_nbre_semaine = int(((_fin - _deb).days // 7) + 1) if (_deb and _fin) else None

schema_simulation = StructType([
    StructField("ID_SIM", StringType(), True),
    StructField("DESC_SIM", StringType(), True),
    StructField("USER", StringType(), True),
    StructField("CREDATE", TimestampType(), True),
    StructField("CAPACITE", StringType(), True),
    StructField("Mode", StringType(), True),
    StructField("NbreSemaine", IntegerType(), True),
    StructField("HorizonDeb", StringType(), True),
    StructField("HorizonFin", StringType(), True),
    StructField("Statut", StringType(), True),
    StructField("EVOL_QTE", StringType(), True),
])

df_simulation = spark.createDataFrame([{
    "ID_SIM": ID_SIM,
    "DESC_SIM": SIM_DESC,
    "USER": SIM_USER,
    "CREDATE": datetime.now(),
    "CAPACITE": CAPACITE_MODE,
    "Mode": MODE_PLANIF,
    "NbreSemaine": _nbre_semaine,
    "HorizonDeb": HORIZON_DEB,
    "HorizonFin": HORIZON_FIN,
    "Statut": "Lancé",
    "EVOL_QTE": EVOL_QTE_DESC,
}], schema=schema_simulation)

df_simulation.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(TBL_SIMULATION)
print(f"[SIM] Table Simulation : ligne ajoutée pour {ID_SIM}")

# ---------------------------------------------------------------------
# EXECUTION DE LA SIMULATION
# ---------------------------------------------------------------------
simulation_mode = (CAPACITE_MODE == "Infinie")
force_infinite_capacity = simulation_mode

all_ofs_scheduled_sim, capture_sim, groups_sim, posts_map_sim = run_planning(
    force_infinite_capacity=force_infinite_capacity,
    simulation_mode=simulation_mode,
    id_sim=ID_SIM,
    log_write_mode="append",
)

write_simulation_outputs(
    all_ofs_scheduled_sim,
    capture_sim,
    groups_sim,
    posts_map_sim,
    write_mode="append",
    id_sim=ID_SIM,
)

print(f"\n[DONE] Simulation {ID_SIM} terminée : tables S_* alimentées en append.")

In [ ]:
# =====================================================================
# EXPORT EXCEL des tables de simulation
# =====================================================================
import os
import pandas as pd

LAKEHOUSE_NAME = "Planning_Lakehouse"


def _detect_table_prefix(spark, probe_table="sim_of_work"):
    for prefix in (f"{LAKEHOUSE_NAME}.", f"{LAKEHOUSE_NAME}.dbo.", ""):
        try:
            spark.table(f"{prefix}{probe_table}").take(1)
            print(f"[Lakehouse] Prefixe detecte : '{prefix}'")
            return prefix
        except Exception:
            continue
    raise RuntimeError(f"Table '{probe_table}' introuvable.")


prefix = _detect_table_prefix(spark, "sim_of_work")

# (nom d'onglet Excel, table Lakehouse)
tables_to_export = [
    ("Simulation",        f"{prefix}Simulation"),
    ("S_grouped_needs",   f"{prefix}S_grouped_needs"),
    ("S_smoothing",       f"{prefix}S_smoothing"),
    ("S_smoothing_ops",   f"{prefix}S_smoothing_operations"),
    ("S_weekly_capacity", f"{prefix}S_weekly_capacity"),
    ("S_groups_anchors",  f"{prefix}S_groups_anchors"),
    ("S_produced_qty",    f"{prefix}S_produced_quantities"),
]

output_folder = "/lakehouse/default/Files/simulation_excel/"
output_path = output_folder + "simulation_data.xlsx"
os.makedirs(output_folder, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for short_name, full_table_name in tables_to_export:
        try:
            print(f"  Lecture de {short_name} ...")
            df_pandas = spark.table(full_table_name).toPandas()
            print(f"    -> {len(df_pandas)} lignes")
            # Excel limite les noms d'onglet a 31 caracteres
            df_pandas.to_excel(writer, sheet_name=short_name[:31], index=False)
        except Exception as e:
            print(f"  ⚠️ Impossible d'exporter {short_name} : {e}")

print(f"✅ Fichier Excel des simulations cree : {output_path}")